In [1]:
"""Report the Kaggle runtime used to design reproducible dependencies."""

from __future__ import annotations

import importlib.metadata
import platform
import sys

import torch

PACKAGE_NAMES = (
    "torch",
    "torchaudio",
    "transformers",
    "speechbrain",
    "wandb",
    "PyYAML",
    "numpy",
    "scipy",
    "scikit-learn",
    "huggingface-hub",
    "safetensors",
)


print("=== SYSTEM ===")
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

print("\n=== PYTORCH AND CUDA ===")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

for device_index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(device_index)
    memory_gib = properties.total_memory / (1024**3)
    print(
        f"GPU {device_index}: {properties.name}, "
        f"compute capability {properties.major}.{properties.minor}, "
        f"{memory_gib:.2f} GiB"
    )

print("\n=== PACKAGE VERSIONS ===")
for package_name in PACKAGE_NAMES:
    try:
        version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        version = "NOT INSTALLED"

    print(f"{package_name}: {version}")

=== SYSTEM ===
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35

=== PYTORCH AND CUDA ===
PyTorch: 2.10.0+cu128
CUDA build: 12.8
CUDA available: True
GPU count: 2
GPU 0: Tesla T4, compute capability 7.5, 14.56 GiB
GPU 1: Tesla T4, compute capability 7.5, 14.56 GiB

=== PACKAGE VERSIONS ===
torch: 2.10.0+cu128
torchaudio: 2.10.0+cu128
transformers: 5.0.0
speechbrain: NOT INSTALLED
wandb: 0.26.1
PyYAML: 6.0.3
numpy: 2.0.2
scipy: 1.16.3
scikit-learn: 1.6.1
huggingface-hub: 1.11.0
safetensors: 0.7.0


In [2]:
!python -m pip install --dry-run \
    "speechbrain==1.1.0" \
    "asteroid-filterbanks==0.4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 3.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 4.7 MB/s eta 0:00:00a 0:00:01
Would install HyperPyYAML-1.2.3 asteroid-filterbanks-0.4.0 ruamel.yaml-0.18.17 ruamel.yaml.clib-0.2.15 speechbrain-1.1.0


In [3]:
"""Install and smoke-test the two model dependencies on Kaggle."""

from __future__ import annotations

import importlib.metadata
import subprocess
import sys

# Install only the packages approved by the dependency dry run.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "speechbrain==1.1.0",
        "asteroid-filterbanks==0.4.0",
    ],
    check=True,
)

# Detect unresolved or incompatible installed-package requirements.
subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    check=True,
)

import torch
from asteroid_filterbanks import Encoder, ParamSincFB
from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("\n=== INSTALLED VERSIONS ===")
print(f"speechbrain: {importlib.metadata.version('speechbrain')}")
print(
    "asteroid-filterbanks: "
    f"{importlib.metadata.version('asteroid-filterbanks')}"
)
print(f"torch: {torch.__version__}")
print(f"device: {device}")

print("\n=== ASTEROID FILTERBANK FORWARD/BACKWARD ===")

# Follow the filterbank construction pattern used by official RawNet3.
filterbank = ParamSincFB(16, 251, stride=10)
encoder = Encoder(filterbank).to(device)

waveforms = torch.randn(2, 1, 16_000, device=device, requires_grad=True)
filterbank_output = encoder(waveforms)
filterbank_output.square().mean().backward()

print(f"output shape: {tuple(filterbank_output.shape)}")
print(f"input gradient available: {waveforms.grad is not None}")

print("\n=== SPEECHBRAIN ECAPA FORWARD/BACKWARD ===")

# A reduced-width ECAPA exercises the official implementation while keeping
# this dependency test inexpensive. Full dimensions are tested later.
ecapa = ECAPA_TDNN(
    input_size=80,
    channels=[64, 64, 64, 64, 192],
    kernel_sizes=[5, 3, 3, 3, 1],
    dilations=[1, 2, 3, 4, 1],
    attention_channels=32,
    lin_neurons=32,
).to(device)

features = torch.randn(2, 100, 80, device=device, requires_grad=True)
embeddings = ecapa(features)
embeddings.square().mean().backward()

print(f"embedding shape: {tuple(embeddings.shape)}")
print(f"input gradient available: {features.grad is not None}")

assert waveforms.grad is not None
assert features.grad is not None
assert torch.isfinite(filterbank_output).all()
assert torch.isfinite(embeddings).all()

print("\nDEPENDENCY SMOKE TEST PASSED")

  Using cached speechbrain-1.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached asteroid_filterbanks-0.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached hyperpyyaml-1.2.3-py3-none-any.whl.metadata (8.1 kB)
  Using cached ruamel_yaml-0.18.17-py3-none-any.whl.metadata (27 kB)
  Using cached ruamel_yaml_clib-0.2.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.5 kB)
Using cached speechbrain-1.1.0-py3-none-any.whl (2.3 MB)
Using cached asteroid_filterbanks-0.4.0-py3-none-any.whl (29 kB)
Using cached hyperpyyaml-1.2.3-py3-none-any.whl (16 kB)
Using cached ruamel_yaml-0.18.17-py3-none-any.whl (121 kB)
Using cached ruamel_yaml_clib-0.2.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (788 kB)
  Attempting uninstall: ruamel.yaml
    Found existing installation: ruamel.yaml 0.19.1
    Uninstalling ruamel.yaml-0.19.1:
      Successfully uninstalled ruamel.yaml-0.19.1
bigframes 2.39.0 requires google-cloud-big

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'check']' returned non-zero exit status 1.

In [4]:
"""Smoke-test the installed model dependencies on the Kaggle GPU."""

from __future__ import annotations

import importlib.metadata

import torch
from asteroid_filterbanks import Encoder, ParamSincFB
from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("=== INSTALLED VERSIONS ===")
print(f"speechbrain: {importlib.metadata.version('speechbrain')}")
print(
    "asteroid-filterbanks: "
    f"{importlib.metadata.version('asteroid-filterbanks')}"
)
print(f"torch: {torch.__version__}")
print(f"torchaudio: {importlib.metadata.version('torchaudio')}")
print(f"device: {device}")

print("\n=== ASTEROID FILTERBANK ===")

filterbank = ParamSincFB(16, 251, stride=10)
encoder = Encoder(filterbank).to(device)

waveforms = torch.randn(
    2,
    1,
    16_000,
    device=device,
    requires_grad=True,
)
filterbank_output = encoder(waveforms)
filterbank_output.square().mean().backward()

print(f"output shape: {tuple(filterbank_output.shape)}")
print(f"finite output: {torch.isfinite(filterbank_output).all().item()}")
print(f"input gradient available: {waveforms.grad is not None}")

print("\n=== SPEECHBRAIN ECAPA-TDNN ===")

# Reduced channels exercise the official architecture inexpensively.
ecapa = ECAPA_TDNN(
    input_size=80,
    channels=[64, 64, 64, 64, 192],
    kernel_sizes=[5, 3, 3, 3, 1],
    dilations=[1, 2, 3, 4, 1],
    attention_channels=32,
    lin_neurons=32,
).to(device)

features = torch.randn(
    2,
    100,
    80,
    device=device,
    requires_grad=True,
)
embeddings = ecapa(features)
embeddings.square().mean().backward()

print(f"embedding shape: {tuple(embeddings.shape)}")
print(f"finite embeddings: {torch.isfinite(embeddings).all().item()}")
print(f"input gradient available: {features.grad is not None}")

assert waveforms.grad is not None
assert features.grad is not None
assert torch.isfinite(filterbank_output).all()
assert torch.isfinite(embeddings).all()

print("\nDEPENDENCY SMOKE TEST PASSED")

=== INSTALLED VERSIONS ===
speechbrain: 1.1.0
asteroid-filterbanks: 0.4.0
torch: 2.10.0+cu128
torchaudio: 2.10.0+cu128
device: cuda:0

=== ASTEROID FILTERBANK ===
output shape: (2, 16, 1575)
finite output: True
input gradient available: True

=== SPEECHBRAIN ECAPA-TDNN ===
embedding shape: (2, 1, 32)
finite embeddings: True
input gradient available: True

DEPENDENCY SMOKE TEST PASSED


In [5]:
"""Smoke-test the exact pretrained SpeechBrain ECAPA-TDNN checkpoint."""

from __future__ import annotations

from pathlib import Path

import torch
import torch.nn.functional as functional
import torchaudio
from huggingface_hub import HfApi, hf_hub_download
from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.fetching import FetchConfig


MODEL_ID = "speechbrain/spkrec-ecapa-voxceleb"
REQUESTED_REVISION = "0f99f2d"
MODEL_DIRECTORY = Path(
    "/kaggle/working/pretrained_models/spkrec-ecapa-voxceleb"
)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"


print("=== MODEL SOURCE ===")

# Resolve the abbreviated revision to its immutable full commit SHA.
model_info = HfApi().model_info(
    repo_id=MODEL_ID,
    revision=REQUESTED_REVISION,
)
resolved_revision = model_info.sha

assert resolved_revision.startswith(REQUESTED_REVISION)

print(f"model ID: {MODEL_ID}")
print(f"requested revision: {REQUESTED_REVISION}")
print(f"resolved revision: {resolved_revision}")
print(f"device: {DEVICE}")


print("\n=== LOAD PRETRAINED MODEL ===")

MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Pin every downloaded model component to the same immutable revision.
fetch_config = FetchConfig(
    revision=resolved_revision,
    allow_updates=True,
)

classifier = EncoderClassifier.from_hparams(
    source=MODEL_ID,
    savedir=str(MODEL_DIRECTORY),
    run_opts={"device": DEVICE},
    fetch_config=fetch_config,
)

parameter_count = sum(
    parameter.numel()
    for parameter in classifier.mods.embedding_model.parameters()
)

print(f"embedding-model parameters: {parameter_count:,}")
print(f"model device: {classifier.device}")


print("\n=== LOAD OFFICIAL SPEECH SAMPLE ===")

sample_path = hf_hub_download(
    repo_id=MODEL_ID,
    filename="example1.wav",
    revision=resolved_revision,
)

waveform, sample_rate = torchaudio.load(sample_path)

# ECAPA expects mono, 16 kHz waveform batches shaped [batch, time].
waveform = waveform.mean(dim=0, keepdim=True)

if sample_rate != 16_000:
    waveform = torchaudio.functional.resample(
        waveform,
        orig_freq=sample_rate,
        new_freq=16_000,
    )
    sample_rate = 16_000

duration_seconds = waveform.shape[-1] / sample_rate

print(f"sample path: {sample_path}")
print(f"sample rate: {sample_rate}")
print(f"waveform shape: {tuple(waveform.shape)}")
print(f"duration: {duration_seconds:.3f} seconds")


print("\n=== EXTRACT EMBEDDING ===")

with torch.inference_mode():
    embedding_1 = classifier.encode_batch(
        waveform,
        normalize=False,
    )
    embedding_2 = classifier.encode_batch(
        waveform,
        normalize=False,
    )

# L2 normalization is the shared representation used for cosine scoring.
normalized_1 = functional.normalize(
    embedding_1.squeeze(1),
    p=2,
    dim=-1,
)
normalized_2 = functional.normalize(
    embedding_2.squeeze(1),
    p=2,
    dim=-1,
)

repeat_similarity = functional.cosine_similarity(
    normalized_1,
    normalized_2,
).item()

print(f"raw embedding shape: {tuple(embedding_1.shape)}")
print(f"normalized shape: {tuple(normalized_1.shape)}")
print(f"finite embedding: {torch.isfinite(embedding_1).all().item()}")
print(f"L2 norm: {normalized_1.norm(dim=-1).item():.6f}")
print(f"repeat cosine similarity: {repeat_similarity:.8f}")


assert embedding_1.shape == (1, 1, 192)
assert torch.isfinite(embedding_1).all()
assert torch.allclose(
    normalized_1.norm(dim=-1),
    torch.ones(1, device=normalized_1.device),
    atol=1e-5,
)
assert repeat_similarity > 0.99999

print("\nPRETRAINED ECAPA CHECKPOINT SMOKE TEST PASSED")

=== MODEL SOURCE ===
model ID: speechbrain/spkrec-ecapa-voxceleb
requested revision: 0f99f2d
resolved revision: 0f99f2d0ebe89ac095bcc5903c4dd8f72b367286
device: cuda:0

=== LOAD PRETRAINED MODEL ===


hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

embedding-model parameters: 20,767,552
model device: cuda:0

=== LOAD OFFICIAL SPEECH SAMPLE ===


example1.wav:   0%|          | 0.00/104k [00:00<?, ?B/s]

sample path: /root/.cache/huggingface/hub/models--speechbrain--spkrec-ecapa-voxceleb/snapshots/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/example1.wav
sample rate: 16000
waveform shape: (1, 52173)
duration: 3.261 seconds

=== EXTRACT EMBEDDING ===
raw embedding shape: (1, 1, 192)
normalized shape: (1, 192)
finite embedding: True
L2 norm: 1.000000
repeat cosine similarity: 1.00000000

PRETRAINED ECAPA CHECKPOINT SMOKE TEST PASSED


In [6]:
"""Download and safely inspect the pinned official RawNet3 checkpoint."""

from __future__ import annotations

import hashlib
from collections.abc import Mapping
from pathlib import Path

import torch
from huggingface_hub import HfApi, hf_hub_download


MODEL_ID = "jungjee/RawNet3"
REQUESTED_REVISION = "c89102eea20c3f96917c434de673c0ace0caddc0"
CHECKPOINT_FILENAME = "model.pt"

# SHA-256 published by the Hugging Face checkpoint page.
EXPECTED_SHA256 = (
    "1ab283bcdf776bfceceea18240e56a8756835b1911b04f9c44f347d47c09f90c"
)


def calculate_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Calculate a file's SHA-256 digest without loading it fully into RAM."""
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


print("=== CHECKPOINT SOURCE ===")

model_info = HfApi().model_info(
    repo_id=MODEL_ID,
    revision=REQUESTED_REVISION,
)
resolved_revision = model_info.sha

assert resolved_revision.startswith(REQUESTED_REVISION)

print(f"model ID: {MODEL_ID}")
print(f"requested revision: {REQUESTED_REVISION}")
print(f"resolved revision: {resolved_revision}")


print("\n=== DOWNLOAD AND INTEGRITY ===")

checkpoint_path = Path(
    hf_hub_download(
        repo_id=MODEL_ID,
        filename=CHECKPOINT_FILENAME,
        revision=resolved_revision,
    )
)

actual_sha256 = calculate_sha256(checkpoint_path)
checkpoint_size_mb = checkpoint_path.stat().st_size / (1024**2)

print(f"checkpoint path: {checkpoint_path}")
print(f"checkpoint size: {checkpoint_size_mb:.2f} MiB")
print(f"expected SHA-256: {EXPECTED_SHA256}")
print(f"actual SHA-256:   {actual_sha256}")
print(f"hash matches: {actual_sha256 == EXPECTED_SHA256}")

assert actual_sha256 == EXPECTED_SHA256


print("\n=== SAFE CHECKPOINT LOAD ===")

# weights_only=True prevents arbitrary checkpoint objects from being executed.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

print(f"top-level type: {type(checkpoint).__name__}")

if isinstance(checkpoint, Mapping):
    print(f"top-level keys: {list(checkpoint.keys())}")
else:
    raise TypeError(
        f"Expected a mapping checkpoint, received {type(checkpoint).__name__}."
    )

# The official trainer stores model weights under the "model" key.
state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint

if not isinstance(state_dict, Mapping):
    raise TypeError("The extracted model state must be a mapping.")

tensor_items = [
    (name, value)
    for name, value in state_dict.items()
    if isinstance(value, torch.Tensor)
]
non_tensor_keys = [
    name
    for name, value in state_dict.items()
    if not isinstance(value, torch.Tensor)
]

total_tensor_elements = sum(tensor.numel() for _, tensor in tensor_items)

print(f"state-dictionary entries: {len(state_dict):,}")
print(f"tensor entries: {len(tensor_items):,}")
print(f"non-tensor entries: {len(non_tensor_keys):,}")
print(f"total tensor elements: {total_tensor_elements:,}")

if non_tensor_keys:
    print(f"non-tensor keys: {non_tensor_keys}")


print("\n=== FIRST 20 STATE-DICTIONARY ENTRIES ===")

for name, tensor in tensor_items[:20]:
    print(
        f"{name}: shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}"
    )

assert tensor_items
assert all(torch.isfinite(tensor).all() for _, tensor in tensor_items)

print("\nRAWNET3 CHECKPOINT INSPECTION PASSED")

=== CHECKPOINT SOURCE ===
model ID: jungjee/RawNet3
requested revision: c89102eea20c3f96917c434de673c0ace0caddc0
resolved revision: c89102eea20c3f96917c434de673c0ace0caddc0

=== DOWNLOAD AND INTEGRITY ===


model.pt:   0%|          | 0.00/65.3M [00:00<?, ?B/s]

checkpoint path: /root/.cache/huggingface/hub/models--jungjee--RawNet3/snapshots/c89102eea20c3f96917c434de673c0ace0caddc0/model.pt
checkpoint size: 62.27 MiB
expected SHA-256: 1ab283bcdf776bfceceea18240e56a8756835b1911b04f9c44f347d47c09f90c
actual SHA-256:   1ab283bcdf776bfceceea18240e56a8756835b1911b04f9c44f347d47c09f90c
hash matches: True

=== SAFE CHECKPOINT LOAD ===
top-level type: dict
top-level keys: ['model']
state-dictionary entries: 234
tensor entries: 234
non-tensor entries: 0
total tensor elements: 16,305,693

=== FIRST 20 STATE-DICTIONARY ENTRIES ===
preprocess.0.flipped_filter: shape=(1, 1, 2), dtype=torch.float32
preprocess.1.weight: shape=(1,), dtype=torch.float32
preprocess.1.bias: shape=(1,), dtype=torch.float32
conv1.filterbank.low_hz_: shape=(128, 1), dtype=torch.float32
conv1.filterbank.band_hz_: shape=(128, 1), dtype=torch.float32
conv1.filterbank.window_: shape=(125,), dtype=torch.float32
conv1.filterbank.n_: shape=(1, 125), dtype=torch.float32
bn1.weight: shape=(

In [7]:
"""Strictly load and smoke-test the official pretrained RawNet3 model."""

from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import torch
import torch.nn.functional as functional
import torchaudio
from huggingface_hub import hf_hub_download


TRAINER_URL = "https://github.com/clovaai/voxceleb_trainer.git"
TRAINER_DIRECTORY = Path(
    "/kaggle/working/vendor/voxceleb_trainer"
)

RAWNET_MODEL_ID = "jungjee/RawNet3"
RAWNET_REVISION = "c89102eea20c3f96917c434de673c0ace0caddc0"

SAMPLE_MODEL_ID = "speechbrain/spkrec-ecapa-voxceleb"
SAMPLE_REVISION = "0f99f2d0ebe89ac095bcc5903c4dd8f72b367286"

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)


def run_command(arguments: list[str], cwd: Path | None = None) -> str:
    """Run a command and return its standard output.

    The argument-list form avoids shell interpolation and makes the executed
    command explicit.
    """
    result = subprocess.run(
        arguments,
        cwd=cwd,
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()


print("=== OFFICIAL ARCHITECTURE SOURCE ===")

if not (TRAINER_DIRECTORY / ".git").is_dir():
    TRAINER_DIRECTORY.parent.mkdir(parents=True, exist_ok=True)

    run_command(
        [
            "git",
            "clone",
            "--depth",
            "1",
            TRAINER_URL,
            str(TRAINER_DIRECTORY),
        ]
    )

remote_url = run_command(
    ["git", "remote", "get-url", "origin"],
    cwd=TRAINER_DIRECTORY,
)
trainer_revision = run_command(
    ["git", "rev-parse", "HEAD"],
    cwd=TRAINER_DIRECTORY,
)
working_tree_status = run_command(
    ["git", "status", "--short"],
    cwd=TRAINER_DIRECTORY,
)

assert remote_url == TRAINER_URL
assert not working_tree_status

print(f"repository: {remote_url}")
print(f"resolved revision: {trainer_revision}")
print("working tree clean: True")


print("\n=== INSTANTIATE RAWNET3 ===")

# The official RawNet3 module imports the accompanying `models` package.
sys.path.insert(0, str(TRAINER_DIRECTORY))

from models.RawNet3 import MainModel  # noqa: E402


model = MainModel(
    nOut=256,
    encoder_type="ECA",
    sinc_stride=10,
).to(DEVICE)

model.eval()

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"device: {DEVICE}")
print(f"parameter count: {parameter_count:,}")
print(f"model state entries: {len(model.state_dict()):,}")


print("\n=== STRICT CHECKPOINT LOAD ===")

checkpoint_path = hf_hub_download(
    repo_id=RAWNET_MODEL_ID,
    filename="model.pt",
    revision=RAWNET_REVISION,
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)
state_dict = checkpoint["model"]

# strict=True proves that every expected name and shape agrees.
load_result = model.load_state_dict(
    state_dict,
    strict=True,
)

print(f"missing keys: {load_result.missing_keys}")
print(f"unexpected keys: {load_result.unexpected_keys}")

assert not load_result.missing_keys
assert not load_result.unexpected_keys


print("\n=== LOAD REAL SPEECH ===")

sample_path = hf_hub_download(
    repo_id=SAMPLE_MODEL_ID,
    filename="example1.wav",
    revision=SAMPLE_REVISION,
)

waveform, sample_rate = torchaudio.load(sample_path)
waveform = waveform.mean(dim=0, keepdim=True)

if sample_rate != 16_000:
    waveform = torchaudio.functional.resample(
        waveform,
        orig_freq=sample_rate,
        new_freq=16_000,
    )
    sample_rate = 16_000

duration_seconds = waveform.shape[-1] / sample_rate

print(f"sample rate: {sample_rate}")
print(f"waveform shape: {tuple(waveform.shape)}")
print(f"duration: {duration_seconds:.3f} seconds")


print("\n=== RAWNET3 GPU INFERENCE ===")

waveform = waveform.to(DEVICE)

with torch.inference_mode():
    embedding_1 = model(waveform)
    embedding_2 = model(waveform)

normalized_1 = functional.normalize(
    embedding_1,
    p=2,
    dim=-1,
)
normalized_2 = functional.normalize(
    embedding_2,
    p=2,
    dim=-1,
)

repeat_similarity = functional.cosine_similarity(
    normalized_1,
    normalized_2,
).item()

print(f"embedding shape: {tuple(embedding_1.shape)}")
print(f"finite embedding: {torch.isfinite(embedding_1).all().item()}")
print(f"L2 norm: {normalized_1.norm(dim=-1).item():.6f}")
print(f"repeat cosine similarity: {repeat_similarity:.8f}")

assert embedding_1.shape == (1, 256)
assert torch.isfinite(embedding_1).all()
assert torch.allclose(
    normalized_1.norm(dim=-1),
    torch.ones(1, device=DEVICE),
    atol=1e-5,
)
assert repeat_similarity > 0.99999

print("\nPRETRAINED RAWNET3 INFERENCE SMOKE TEST PASSED")

=== OFFICIAL ARCHITECTURE SOURCE ===
repository: https://github.com/clovaai/voxceleb_trainer.git
resolved revision: f51bab870672a9b0b50fa158b4e30f329e7866d7
working tree clean: True

=== INSTANTIATE RAWNET3 ===
self.encoder_type ECA
device: cuda:0
parameter count: 16,280,322
model state entries: 234

=== STRICT CHECKPOINT LOAD ===
missing keys: []
unexpected keys: []

=== LOAD REAL SPEECH ===
sample rate: 16000
waveform shape: (1, 52173)
duration: 3.261 seconds

=== RAWNET3 GPU INFERENCE ===
embedding shape: (1, 256)
finite embedding: True
L2 norm: 1.000000
repeat cosine similarity: 1.00000000

PRETRAINED RAWNET3 INFERENCE SMOKE TEST PASSED


In [8]:
"""Audit the pinned WavLM+MHFA repository without executing its source code."""

from __future__ import annotations

from pathlib import Path

import yaml
from huggingface_hub import HfApi, hf_hub_download


MODEL_ID = "theolepage/wavlm_ssl_sv"
REQUESTED_REVISION = "bfb8527de83b5347fb81b1e9e31be241656ca103"

CONFIG_PATH = "configs/wavlm_mhfa_dlg_lc.yaml"
README_PATH = "README.md"

WEIGHT_EXTENSIONS = {
    ".pt",
    ".pth",
    ".ckpt",
    ".bin",
    ".safetensors",
}


print("=== REPOSITORY SOURCE ===")

api = HfApi()
model_info = api.model_info(
    repo_id=MODEL_ID,
    revision=REQUESTED_REVISION,
    files_metadata=True,
)
resolved_revision = model_info.sha

assert resolved_revision == REQUESTED_REVISION

print(f"model ID: {MODEL_ID}")
print(f"requested revision: {REQUESTED_REVISION}")
print(f"resolved revision: {resolved_revision}")


print("\n=== COMPLETE FILE INVENTORY ===")

repository_files = sorted(
    model_info.siblings,
    key=lambda sibling: sibling.rfilename,
)

known_sizes = [
    sibling.size
    for sibling in repository_files
    if sibling.size is not None
]
total_known_size = sum(known_sizes)

for sibling in repository_files:
    size_text = (
        f"{sibling.size:,} bytes"
        if sibling.size is not None
        else "size unavailable"
    )
    print(f"{sibling.rfilename}: {size_text}")

print(f"\nfile count: {len(repository_files)}")
print(f"known repository size: {total_known_size / 1024:.2f} KiB")


print("\n=== PACKAGED WEIGHT SEARCH ===")

weight_files = [
    sibling.rfilename
    for sibling in repository_files
    if Path(sibling.rfilename).suffix.lower() in WEIGHT_EXTENSIONS
]

print(f"packaged weight files: {weight_files}")
print(f"packaged weight count: {len(weight_files)}")


print("\n=== REQUIRED SOURCE FILES ===")

available_paths = {
    sibling.rfilename
    for sibling in repository_files
}

required_paths = {
    README_PATH,
    CONFIG_PATH,
    "models/Baseline/Spk_Encoder.py",
    "models/Baseline/WavLM.py",
}

for required_path in sorted(required_paths):
    exists = required_path in available_paths
    print(f"{required_path}: {exists}")
    assert exists


print("\n=== OFFICIAL BASE CONFIGURATION ===")

config_local_path = hf_hub_download(
    repo_id=MODEL_ID,
    filename=CONFIG_PATH,
    revision=resolved_revision,
)

with open(config_local_path, encoding="utf-8") as config_file:
    configuration = yaml.safe_load(config_file)

print(
    yaml.safe_dump(
        configuration,
        sort_keys=True,
        default_flow_style=False,
    )
)


print("=== README ARTIFACT REFERENCES ===")

readme_local_path = hf_hub_download(
    repo_id=MODEL_ID,
    filename=README_PATH,
    revision=resolved_revision,
)

with open(readme_local_path, encoding="utf-8") as readme_file:
    readme_text = readme_file.read()

keywords = (
    "download",
    "checkpoint",
    "weight",
    "wavlm-base+",
    "dino",
)

relevant_lines = [
    line.strip()
    for line in readme_text.splitlines()
    if line.strip()
    and any(keyword in line.lower() for keyword in keywords)
]

for line in relevant_lines:
    print(line)


# At this pinned revision, the repository is expected to contain source and
# configuration only, while model artifacts are referenced externally.
assert not weight_files
assert "WavLM-Base+.pt" in readme_text
assert configuration["model"] == "Baseline"

print("\nWAVLM+MHFA REPOSITORY AUDIT PASSED")

=== REPOSITORY SOURCE ===
model ID: theolepage/wavlm_ssl_sv
requested revision: bfb8527de83b5347fb81b1e9e31be241656ca103
resolved revision: bfb8527de83b5347fb81b1e9e31be241656ca103

=== COMPLETE FILE INVENTORY ===
.gitattributes: 1,519 bytes
.gitignore: 44 bytes
DatasetLoader.py: 11,054 bytes
LICENSE.md: 1,068 bytes
README.md: 4,199 bytes
SpeakerNet.py: 13,546 bytes
configs/wavlm_mhfa_dlg_lc.yaml: 476 bytes
configs/wavlm_mhfa_dlg_lc_lmft.yaml: 447 bytes
loss/aamsoftmax.py: 4,943 bytes
models/Baseline/Spk_Encoder.py: 4,009 bytes
models/Baseline/WavLM.py: 27,528 bytes
models/Baseline/modules.py: 31,946 bytes
optimizer/adamw.py: 188 bytes
pseudo_labeling.py: 2,204 bytes
requirements.txt: 133 bytes
scheduler/steplr.py: 317 bytes
tools/rsync_jz.sh: 519 bytes
trainSpeakerNet.py: 16,188 bytes
trainSpeakerNet_Eval.py: 11,577 bytes
train_ddp_jz.sh: 461 bytes
training_framework.svg: 743,874 bytes
tuneThreshold.py: 3,392 bytes
utils.py: 1,285 bytes

file count: 23
known repository size: 860.27 Ki

wavlm_mhfa_dlg_lc.yaml:   0%|          | 0.00/476 [00:00<?, ?B/s]

LLRD_factor: 1.0
LR_MHFA: 5e-3
LR_Transformer: 2e-5
augment: true
batch_size: 120
eval_frames: 400
lr_decay: 0.95
margin: 0.2
max_epoch: 15
max_frames: 300
model: Baseline.Spk_Encoder
nClasses: 7500
nOut: 256
port: 6754
pretrained_model_path: WavLM-Base+.pt
save_path: exp/wavlm_mhfa_dlg_lc
scale: 30
trainfunc: aamsoftmax
weight_finetuning_reg: 0.01

=== README ARTIFACT REFERENCES ===


README.md: 0.00B [00:00, ?B/s]

- DINO
The proposed framework fine-tunes a pre-trained **WavLM** using pseudo-labels, generated through **Self-Supervised Learning** (SSL), for **Speaker Verification** (SV). Initial pseudo-labels are derived from an SSL DINO-based model and are iteratively refined by clustering the model embeddings.
- Download [WavLM-Base+ model](https://github.com/microsoft/unilm/tree/master/wavlm) and place `WavLM-Base+.pt` at the root folder.
#### Step 1: Extract DINO speaker embeddings
The code to train the DINO model is not currently provided. We recommend using [sslsv](https://github.com/theolepage/sslsv) or [3D-Speaker](https://github.com/modelscope/3D-Speaker) to extract initial speaker embeddings.
Alternatively, you can directly download the DINO embeddings we used for our system: [dino_vox2_embeddings.pt](https://drive.google.com/file/d/1YnxrMIgrr6NQgZ3Hv2_5YdP5W8xfdyLH/view?usp=sharing).
1. Copy the latest model checkpoint to `exp/wavlm_mhfa_dlg_lc_lmft/model` to resume training.
### Model 

AssertionError: 

In [1]:
"""Download WavLM-Base+ from Microsoft's official Google Drive mirror."""

from __future__ import annotations

import importlib.metadata
import importlib.util
import subprocess
import sys
from pathlib import Path


GOOGLE_DRIVE_FILE_ID = "1-zlAj2SyVJVsbhifwpTlAfrgc9qu-HDb"
CHECKPOINT_PATH = Path(
    "/kaggle/working/artifacts/wavlm/WavLM-Base+.pt"
)
PARTIAL_PATH = CHECKPOINT_PATH.with_suffix(".pt.part")


if importlib.util.find_spec("gdown") is None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "gdown==5.2.0",
        ],
        check=True,
    )

import gdown  # noqa: E402


print(f"gdown version: {importlib.metadata.version('gdown')}")

CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
PARTIAL_PATH.unlink(missing_ok=True)

downloaded_path = gdown.download(
    id=GOOGLE_DRIVE_FILE_ID,
    output=str(PARTIAL_PATH),
    quiet=False,
)

if downloaded_path is None or not PARTIAL_PATH.is_file():
    raise RuntimeError("Google Drive checkpoint download failed.")

downloaded_size_mib = PARTIAL_PATH.stat().st_size / (1024**2)

# A lower bound prevents an HTML error page from being accepted as a model.
if downloaded_size_mib < 100:
    raise RuntimeError(
        f"Downloaded file is unexpectedly small: "
        f"{downloaded_size_mib:.2f} MiB."
    )

PARTIAL_PATH.replace(CHECKPOINT_PATH)

print(f"checkpoint path: {CHECKPOINT_PATH}")
print(f"checkpoint size: {downloaded_size_mib:.2f} MiB")
print("\nWAVLM-BASE+ GOOGLE DRIVE DOWNLOAD PASSED")

gdown version: 5.2.2


Downloading...
From (original): https://drive.google.com/uc?id=1-zlAj2SyVJVsbhifwpTlAfrgc9qu-HDb
From (redirected): https://drive.google.com/uc?id=1-zlAj2SyVJVsbhifwpTlAfrgc9qu-HDb&confirm=t&uuid=9ee4f328-52ed-489c-ad9e-7c309c02094a
To: /kaggle/working/artifacts/wavlm/WavLM-Base+.pt.part
100%|██████████| 378M/378M [00:02<00:00, 151MB/s]  

checkpoint path: /kaggle/working/artifacts/wavlm/WavLM-Base+.pt
checkpoint size: 360.11 MiB

WAVLM-BASE+ GOOGLE DRIVE DOWNLOAD PASSED


In [2]:
"""Download and inspect Microsoft's official WavLM-Base+ checkpoint."""

from __future__ import annotations

import hashlib
from collections.abc import Mapping
from pathlib import Path

import requests
import torch
from tqdm.auto import tqdm


DOWNLOAD_URL = (
    "https://valle.blob.core.windows.net/share/wavlm/WavLM-Base+.pt"
    "?sv=2021-10-04"
    "&st=2024-04-04T07%3A15%3A11Z"
    "&se=2034-04-05T07%3A15%3A00Z"
    "&sr=c"
    "&sp=rl"
    "&sig=xH3MbkMqHPLBI5gN%2Frt9H4J8Ai%2BtUnkduo7KGpkLbdA%3D"
)

CHECKPOINT_PATH = Path(
    "/kaggle/working/artifacts/wavlm/WavLM-Base+.pt"
)


def calculate_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Calculate a SHA-256 digest without loading the file into RAM."""
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


def download_file(url: str, destination: Path) -> None:
    """Download a file atomically with progress reporting.

    A temporary file prevents an interrupted download from being mistaken for
    a complete checkpoint.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial_path = destination.with_suffix(destination.suffix + ".part")
    partial_path.unlink(missing_ok=True)

    with requests.get(
        url,
        stream=True,
        timeout=(30, 600),
    ) as response:
        response.raise_for_status()

        total_bytes = int(
            response.headers.get("content-length", 0)
        )

        with partial_path.open("wb") as output_file:
            with tqdm(
                total=total_bytes,
                unit="B",
                unit_scale=True,
                desc=destination.name,
            ) as progress:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        output_file.write(chunk)
                        progress.update(len(chunk))

    partial_path.replace(destination)


print("=== OFFICIAL ARTIFACT DOWNLOAD ===")

if CHECKPOINT_PATH.exists():
    print("Using the existing completed checkpoint.")
else:
    download_file(DOWNLOAD_URL, CHECKPOINT_PATH)

checkpoint_size_mib = CHECKPOINT_PATH.stat().st_size / (1024**2)
checkpoint_sha256 = calculate_sha256(CHECKPOINT_PATH)

print(f"path: {CHECKPOINT_PATH}")
print(f"size: {checkpoint_size_mib:.2f} MiB")
print(f"SHA-256: {checkpoint_sha256}")


print("\n=== PICKLE GLOBAL INSPECTION ===")

# PyTorch reports objects that may prevent restricted weights-only loading.
unsafe_globals = (
    torch.serialization.get_unsafe_globals_in_checkpoint(
        CHECKPOINT_PATH
    )
)

print(f"unsafe global count: {len(unsafe_globals)}")

for global_name in unsafe_globals:
    print(global_name)


print("\n=== RESTRICTED CHECKPOINT LOAD ===")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
    mmap=True,
)

if not isinstance(checkpoint, Mapping):
    raise TypeError(
        f"Expected a mapping, received {type(checkpoint).__name__}."
    )

print(f"top-level type: {type(checkpoint).__name__}")
print(f"top-level keys: {list(checkpoint.keys())}")

assert "cfg" in checkpoint
assert "model" in checkpoint


print("\n=== WAVLM CONFIGURATION ===")

configuration = checkpoint["cfg"]

if isinstance(configuration, Mapping):
    configuration_items = dict(configuration)
elif hasattr(configuration, "__dict__"):
    configuration_items = vars(configuration)
else:
    configuration_items = {}

print(f"configuration type: {type(configuration).__name__}")
print(f"configuration key count: {len(configuration_items)}")

for key in sorted(configuration_items):
    print(f"{key}: {configuration_items[key]}")


print("\n=== MODEL STATE DICTIONARY ===")

state_dict = checkpoint["model"]

if not isinstance(state_dict, Mapping):
    raise TypeError("The model state must be a mapping.")

tensor_items = [
    (name, value)
    for name, value in state_dict.items()
    if isinstance(value, torch.Tensor)
]

non_tensor_keys = [
    name
    for name, value in state_dict.items()
    if not isinstance(value, torch.Tensor)
]

total_tensor_elements = sum(
    tensor.numel()
    for _, tensor in tensor_items
)

print(f"state entries: {len(state_dict):,}")
print(f"tensor entries: {len(tensor_items):,}")
print(f"non-tensor entries: {len(non_tensor_keys):,}")
print(f"total tensor elements: {total_tensor_elements:,}")


print("\n=== FIRST 15 MODEL TENSORS ===")

for name, tensor in tensor_items[:15]:
    print(
        f"{name}: shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}"
    )

assert tensor_items
assert not non_tensor_keys

print("\nWAVLM-BASE+ CHECKPOINT INSPECTION PASSED")

=== OFFICIAL ARTIFACT DOWNLOAD ===
Using the existing completed checkpoint.
path: /kaggle/working/artifacts/wavlm/WavLM-Base+.pt
size: 360.11 MiB
SHA-256: fcbcf2a94def92e90e086bb0727275d53b75a9c0e483e2abfa560ac951986b6d

=== PICKLE GLOBAL INSPECTION ===
unsafe global count: 0

=== RESTRICTED CHECKPOINT LOAD ===
top-level type: dict
top-level keys: ['cfg', 'model']

=== WAVLM CONFIGURATION ===
configuration type: dict
configuration key count: 35
activation_dropout: 0.0
activation_fn: gelu
attention_dropout: 0.1
conv_bias: False
conv_feature_layers: [(512,10,5)] + [(512,3,2)] * 4 + [(512,2,2)] * 2
conv_pos: 128
conv_pos_groups: 16
dropout: 0.1
dropout_features: 0.1
dropout_input: 0.1
encoder_attention_heads: 12
encoder_embed_dim: 768
encoder_ffn_embed_dim: 3072
encoder_layerdrop: 0.05
encoder_layers: 12
extractor_mode: default
feature_grad_mult: 0.1
gru_rel_pos: True
layer_norm_first: False
mask_channel_length: 10
mask_channel_min_space: 1
mask_channel_other: 0.0
mask_channel_prob: 0.0
m

In [3]:
"""Strictly load WavLM-Base+ and test the required MHFA architecture."""

from __future__ import annotations

import gc
import sys
from pathlib import Path

import torch
import torch.nn.functional as functional
import torchaudio
from huggingface_hub import (
    HfApi,
    hf_hub_download,
    snapshot_download,
)


SOURCE_MODEL_ID = "theolepage/wavlm_ssl_sv"
SOURCE_REVISION = "bfb8527de83b5347fb81b1e9e31be241656ca103"
SOURCE_DIRECTORY = Path(
    "/kaggle/working/vendor/wavlm_ssl_sv"
)

WAVLM_CHECKPOINT_PATH = Path(
    "/kaggle/working/artifacts/wavlm/WavLM-Base+.pt"
)

SAMPLE_MODEL_ID = "speechbrain/spkrec-ecapa-voxceleb"
SAMPLE_REVISION = "0f99f2d0ebe89ac095bcc5903c4dd8f72b367286"

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)


print("=== PINNED ARCHITECTURE SOURCE ===")

source_info = HfApi().model_info(
    repo_id=SOURCE_MODEL_ID,
    revision=SOURCE_REVISION,
)

assert source_info.sha == SOURCE_REVISION

snapshot_download(
    repo_id=SOURCE_MODEL_ID,
    revision=SOURCE_REVISION,
    local_dir=SOURCE_DIRECTORY,
    allow_patterns=[
        "models/Baseline/*.py",
        "LICENSE.md",
    ],
)

required_source_files = [
    SOURCE_DIRECTORY / "models/Baseline/Spk_Encoder.py",
    SOURCE_DIRECTORY / "models/Baseline/WavLM.py",
    SOURCE_DIRECTORY / "models/Baseline/modules.py",
]

for source_file in required_source_files:
    relative_path = source_file.relative_to(SOURCE_DIRECTORY)
    exists = source_file.is_file()

    print(f"{relative_path}: {exists}")
    assert exists

print(f"repository revision: {source_info.sha}")


print("\n=== IMPORT REQUIRED COMPONENTS ===")

sys.path.insert(0, str(SOURCE_DIRECTORY))

from models.Baseline.Spk_Encoder import MHFA  # noqa: E402
from models.Baseline.WavLM import WavLM, WavLMConfig  # noqa: E402


print("WavLM import: passed")
print("MHFA import: passed")


print("\n=== RESTRICTED CHECKPOINT LOAD ===")

assert WAVLM_CHECKPOINT_PATH.is_file(), (
    f"Checkpoint not found: {WAVLM_CHECKPOINT_PATH}"
)

checkpoint = torch.load(
    WAVLM_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
    mmap=True,
)

wavlm_configuration = WavLMConfig(checkpoint["cfg"])
wavlm = WavLM(wavlm_configuration)

# Strict loading proves that every parameter name and shape agrees.
load_result = wavlm.load_state_dict(
    checkpoint["model"],
    strict=True,
)

print(f"missing keys: {load_result.missing_keys}")
print(f"unexpected keys: {load_result.unexpected_keys}")

assert not load_result.missing_keys
assert not load_result.unexpected_keys


print("\n=== BUILD MHFA SPEAKER ENCODER ===")

# The required implementation combines 13 representation levels with
# a 64-head Multi-Head Factorized Attentive Pooling backend.
mhfa = MHFA(
    head_nb=64,
    inputs_dim=768,
    compression_dim=128,
    outputs_dim=256,
)

wavlm_parameter_count = sum(
    parameter.numel()
    for parameter in wavlm.parameters()
)
mhfa_parameter_count = sum(
    parameter.numel()
    for parameter in mhfa.parameters()
)

print(f"WavLM parameters: {wavlm_parameter_count:,}")
print(f"MHFA parameters: {mhfa_parameter_count:,}")
print(
    "total parameters: "
    f"{wavlm_parameter_count + mhfa_parameter_count:,}"
)

wavlm = wavlm.to(DEVICE).eval()
mhfa = mhfa.to(DEVICE).eval()

# The weights have been copied into the model, so release checkpoint references.
del checkpoint
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("\n=== LOAD REAL SPEECH ===")

sample_path = hf_hub_download(
    repo_id=SAMPLE_MODEL_ID,
    filename="example1.wav",
    revision=SAMPLE_REVISION,
)

waveform, sample_rate = torchaudio.load(sample_path)
waveform = waveform.mean(dim=0, keepdim=True)

if sample_rate != 16_000:
    waveform = torchaudio.functional.resample(
        waveform,
        orig_freq=sample_rate,
        new_freq=16_000,
    )
    sample_rate = 16_000

waveform = waveform.to(DEVICE)
duration_seconds = waveform.shape[-1] / sample_rate

print(f"sample rate: {sample_rate}")
print(f"waveform shape: {tuple(waveform.shape)}")
print(f"duration: {duration_seconds:.3f} seconds")


def extract_mhfa_embedding(
    waveform_batch: torch.Tensor,
) -> tuple[torch.Tensor, int, tuple[int, ...]]:
    """Extract an embedding using the official WavLM+MHFA tensor layout."""
    _, layer_results = wavlm.extract_features(
        waveform_batch,
        output_layer=13,
    )

    # Convert each level from [time, batch, dimension] to
    # [batch, time, dimension].
    layer_representations = [
        representation.transpose(0, 1)
        for representation, _ in layer_results
    ]

    # Reproduce Spk_Encoder.py:
    # [layers, batch, time, dimension]
    # -> [batch, dimension, time, layers].
    stacked_features = (
        torch.stack(layer_representations)
        .transpose(0, -1)
        .transpose(0, 1)
    )

    embedding = mhfa(stacked_features)

    return (
        embedding,
        len(layer_representations),
        tuple(stacked_features.shape),
    )


print("\n=== WAVLM+MHFA GPU INFERENCE ===")

with torch.inference_mode():
    embedding_1, layer_count, stacked_shape = (
        extract_mhfa_embedding(waveform)
    )
    embedding_2, _, _ = extract_mhfa_embedding(waveform)

normalized_1 = functional.normalize(
    embedding_1,
    p=2,
    dim=-1,
)
normalized_2 = functional.normalize(
    embedding_2,
    p=2,
    dim=-1,
)

repeat_similarity = functional.cosine_similarity(
    normalized_1,
    normalized_2,
).item()

print(f"representation levels: {layer_count}")
print(f"stacked MHFA input shape: {stacked_shape}")
print(f"embedding shape: {tuple(embedding_1.shape)}")
print(
    "finite embedding: "
    f"{torch.isfinite(embedding_1).all().item()}"
)
print(f"L2 norm: {normalized_1.norm(dim=-1).item():.6f}")
print(f"repeat cosine similarity: {repeat_similarity:.8f}")

assert layer_count == 13
assert stacked_shape[0] == 1
assert stacked_shape[1] == 768
assert stacked_shape[-1] == 13
assert embedding_1.shape == (1, 256)
assert torch.isfinite(embedding_1).all()
assert repeat_similarity > 0.99999

print("\nWAVLM+MHFA ARCHITECTURE SMOKE TEST PASSED")

=== PINNED ARCHITECTURE SOURCE ===


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

models/Baseline/Spk_Encoder.py: True
models/Baseline/WavLM.py: True
models/Baseline/modules.py: True
repository revision: bfb8527de83b5347fb81b1e9e31be241656ca103

=== IMPORT REQUIRED COMPONENTS ===
WavLM import: passed
MHFA import: passed

=== RESTRICTED CHECKPOINT LOAD ===


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


missing keys: []
unexpected keys: []

=== BUILD MHFA SPEAKER ENCODER ===
WavLM parameters: 94,381,936
MHFA parameters: 2,302,554
total parameters: 96,684,490

=== LOAD REAL SPEECH ===


example1.wav:   0%|          | 0.00/104k [00:00<?, ?B/s]

sample rate: 16000
waveform shape: (1, 52173)
duration: 3.261 seconds

=== WAVLM+MHFA GPU INFERENCE ===
representation levels: 13
stacked MHFA input shape: (1, 768, 162, 13)
embedding shape: (1, 256)
finite embedding: True
L2 norm: 1.000000
repeat cosine similarity: 1.00000012

WAVLM+MHFA ARCHITECTURE SMOKE TEST PASSED


In [1]:
"""Inspect the top-level layout of every attached Kaggle dataset."""

from __future__ import annotations

from pathlib import Path


INPUT_ROOT = Path("/kaggle/input")
MAX_ENTRIES_PER_DATASET = 40


print("=== KAGGLE INPUT ROOT ===")
print(f"path: {INPUT_ROOT}")
print(f"exists: {INPUT_ROOT.is_dir()}")

assert INPUT_ROOT.is_dir()


dataset_directories = sorted(
    path
    for path in INPUT_ROOT.iterdir()
    if path.is_dir()
)

print(f"attached dataset count: {len(dataset_directories)}")
print(
    "attached datasets: "
    f"{[path.name for path in dataset_directories]}"
)


for dataset_directory in dataset_directories:
    print(f"\n=== DATASET: {dataset_directory.name} ===")

    entries = sorted(
        dataset_directory.iterdir(),
        key=lambda path: (not path.is_dir(), path.name.lower()),
    )

    print(f"top-level entry count: {len(entries)}")

    for entry in entries[:MAX_ENTRIES_PER_DATASET]:
        entry_type = "directory" if entry.is_dir() else "file"

        if entry.is_file():
            size_mib = entry.stat().st_size / (1024**2)
            print(
                f"[{entry_type}] {entry.name} "
                f"({size_mib:.2f} MiB)"
            )
        else:
            print(f"[{entry_type}] {entry.name}")

    omitted_count = max(
        0,
        len(entries) - MAX_ENTRIES_PER_DATASET,
    )

    if omitted_count:
        print(f"... {omitted_count} additional entries omitted")


print("\nKAGGLE DATASET MOUNT DISCOVERY PASSED")

=== KAGGLE INPUT ROOT ===
path: /kaggle/input
exists: True
attached dataset count: 1
attached datasets: ['datasets']

=== DATASET: datasets ===
top-level entry count: 1
[directory] dullahn

KAGGLE DATASET MOUNT DISCOVERY PASSED


In [2]:
"""Discover datasets inside Kaggle's publisher-based mount structure."""

from __future__ import annotations

from pathlib import Path


INPUT_ROOT = Path("/kaggle/input")
PUBLISHER_ROOT = INPUT_ROOT / "datasets" / "dullahn"
MAX_ENTRIES = 25


print("=== PUBLISHER ROOT ===")
print(f"path: {PUBLISHER_ROOT}")
print(f"exists: {PUBLISHER_ROOT.is_dir()}")

assert PUBLISHER_ROOT.is_dir()


dataset_roots = sorted(
    path
    for path in PUBLISHER_ROOT.iterdir()
    if path.is_dir()
)

print(f"attached dataset count: {len(dataset_roots)}")
print(
    "dataset mount names: "
    f"{[path.name for path in dataset_roots]}"
)

assert len(dataset_roots) == 2


for dataset_root in dataset_roots:
    print(f"\n=== DATASET: {dataset_root.name} ===")

    # Kaggle commonly inserts a versions/<number> directory.
    current_paths = [dataset_root]

    for depth in range(3):
        next_paths: list[Path] = []

        for current_path in current_paths:
            entries = sorted(
                current_path.iterdir(),
                key=lambda path: (
                    not path.is_dir(),
                    path.name.lower(),
                ),
            )

            relative_path = current_path.relative_to(
                PUBLISHER_ROOT
            )

            print(f"\n[{relative_path}]")
            print(f"entry count: {len(entries)}")

            for entry in entries[:MAX_ENTRIES]:
                if entry.is_dir():
                    print(f"  [directory] {entry.name}")
                    next_paths.append(entry)
                else:
                    size_mib = entry.stat().st_size / (1024**2)
                    print(
                        f"  [file] {entry.name} "
                        f"({size_mib:.2f} MiB)"
                    )

            omitted = max(0, len(entries) - MAX_ENTRIES)

            if omitted:
                print(f"  ... {omitted} entries omitted")

        # Avoid descending into many speaker directories during discovery.
        current_paths = next_paths[:4]

        if not current_paths:
            break


print("\nNESTED KAGGLE DATASET DISCOVERY PASSED")

=== PUBLISHER ROOT ===
path: /kaggle/input/datasets/dullahn
exists: True
attached dataset count: 2
dataset mount names: ['mozzila-tidyvoice', 'vimd-dataset']

=== DATASET: mozzila-tidyvoice ===

[mozzila-tidyvoice]
entry count: 1
  [directory] TidyVoiceX_ASV

[mozzila-tidyvoice/TidyVoiceX_ASV]
entry count: 2
  [directory] TidyVoiceX_Dev
  [directory] TidyVoiceX_Train

[mozzila-tidyvoice/TidyVoiceX_ASV/TidyVoiceX_Dev]
entry count: 1
  [directory] TidyVoiceX_Dev

[mozzila-tidyvoice/TidyVoiceX_ASV/TidyVoiceX_Train]
entry count: 1
  [directory] TidyVoiceX_Train

=== DATASET: vimd-dataset ===

[vimd-dataset]
entry count: 4
  [directory] .cache
  [directory] data
  [file] .gitattributes (0.00 MiB)
  [file] README.md (0.01 MiB)

[vimd-dataset/.cache]
entry count: 1
  [directory] huggingface

[vimd-dataset/data]
entry count: 130
  [file] test-00000-of-00014.parquet (410.17 MiB)
  [file] test-00001-of-00014.parquet (447.92 MiB)
  [file] test-00002-of-00014.parquet (399.26 MiB)
  [file] test-000

In [3]:
"""Audit TidyVoice files and ViMD Parquet metadata without loading audio."""

from __future__ import annotations

import os
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pyarrow as arrow
import pyarrow.parquet as parquet


PUBLISHER_ROOT = Path("/kaggle/input/datasets/dullahn")

TIDYVOICE_ROOT = (
    PUBLISHER_ROOT
    / "mozzila-tidyvoice"
    / "TidyVoiceX_ASV"
)
VIMD_ROOT = PUBLISHER_ROOT / "vimd-dataset"
VIMD_DATA_ROOT = VIMD_ROOT / "data"

AUDIO_EXTENSIONS = {
    ".wav",
    ".flac",
    ".mp3",
    ".ogg",
    ".m4a",
    ".opus",
}
MANIFEST_EXTENSIONS = {
    ".csv",
    ".tsv",
    ".txt",
    ".json",
    ".jsonl",
    ".scp",
    ".lst",
}


def infer_tidyvoice_split(path: Path) -> str:
    """Infer the provided TidyVoice branch from its path components."""
    lowercase_parts = {
        part.lower()
        for part in path.parts
    }

    if any("train" in part for part in lowercase_parts):
        return "train"

    if any("dev" in part for part in lowercase_parts):
        return "dev"

    return "unknown"


def contains_binary(data_type: arrow.DataType) -> bool:
    """Return whether an Arrow type directly or recursively contains bytes."""
    if (
        arrow.types.is_binary(data_type)
        or arrow.types.is_large_binary(data_type)
        or arrow.types.is_fixed_size_binary(data_type)
    ):
        return True

    if arrow.types.is_struct(data_type):
        return any(
            contains_binary(field.type)
            for field in data_type
        )

    if (
        arrow.types.is_list(data_type)
        or arrow.types.is_large_list(data_type)
        or arrow.types.is_fixed_size_list(data_type)
    ):
        return contains_binary(data_type.value_type)

    return False


def truncate_value(value: Any, limit: int = 180) -> Any:
    """Shorten preview values so dataset inspection remains readable."""
    if isinstance(value, bytes):
        return f"<{len(value):,} bytes>"

    if isinstance(value, str) and len(value) > limit:
        return value[:limit] + "..."

    if isinstance(value, dict):
        return {
            key: truncate_value(item, limit)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            truncate_value(item, limit)
            for item in value[:10]
        ]

    return value


print("=== TIDYVOICE FILE INVENTORY ===")
print(f"root: {TIDYVOICE_ROOT}")
print(f"exists: {TIDYVOICE_ROOT.is_dir()}")

assert TIDYVOICE_ROOT.is_dir()

extension_counts: Counter[str] = Counter()
split_file_counts: Counter[str] = Counter()
split_audio_counts: Counter[str] = Counter()
audio_samples: dict[str, list[str]] = defaultdict(list)
manifest_paths: list[Path] = []
total_files = 0

# os.walk streams directory entries instead of building a complete path list.
for directory_path, _, filenames in os.walk(TIDYVOICE_ROOT):
    directory = Path(directory_path)

    for filename in filenames:
        file_path = directory / filename
        relative_path = file_path.relative_to(TIDYVOICE_ROOT)
        extension = file_path.suffix.lower() or "<no extension>"
        split = infer_tidyvoice_split(relative_path)

        total_files += 1
        extension_counts[extension] += 1
        split_file_counts[split] += 1

        if extension in AUDIO_EXTENSIONS:
            split_audio_counts[split] += 1

            if len(audio_samples[split]) < 8:
                audio_samples[split].append(str(relative_path))

        if extension in MANIFEST_EXTENSIONS:
            manifest_paths.append(relative_path)

print(f"total files: {total_files:,}")
print(f"files by branch: {dict(split_file_counts)}")
print(f"audio files by branch: {dict(split_audio_counts)}")

print("\nfile extensions:")
for extension, count in extension_counts.most_common():
    print(f"  {extension}: {count:,}")

print("\nmanifest-like files:")
for path in manifest_paths[:30]:
    print(f"  {path}")

if len(manifest_paths) > 30:
    print(
        f"  ... {len(manifest_paths) - 30} "
        "additional manifest-like files"
    )

print("\nrepresentative audio paths:")
for split, paths in sorted(audio_samples.items()):
    print(f"  [{split}]")

    for path in paths:
        print(f"    {path}")


print("\n=== VIMD PARQUET INVENTORY ===")
print(f"root: {VIMD_DATA_ROOT}")
print(f"exists: {VIMD_DATA_ROOT.is_dir()}")

assert VIMD_DATA_ROOT.is_dir()

parquet_files = sorted(VIMD_DATA_ROOT.glob("*.parquet"))

assert parquet_files

split_shard_counts: Counter[str] = Counter()
split_row_counts: Counter[str] = Counter()
split_size_bytes: Counter[str] = Counter()
reference_schemas: dict[str, arrow.Schema] = {}
schema_mismatches: list[str] = []

for index, parquet_path in enumerate(parquet_files, start=1):
    split = parquet_path.name.split("-", maxsplit=1)[0]
    parquet_file = parquet.ParquetFile(parquet_path)
    schema = parquet_file.schema_arrow.remove_metadata()

    split_shard_counts[split] += 1
    split_row_counts[split] += parquet_file.metadata.num_rows
    split_size_bytes[split] += parquet_path.stat().st_size

    if split not in reference_schemas:
        reference_schemas[split] = schema
    elif not schema.equals(
        reference_schemas[split],
        check_metadata=False,
    ):
        schema_mismatches.append(parquet_path.name)

    if index % 25 == 0 or index == len(parquet_files):
        print(
            f"inspected {index}/{len(parquet_files)} "
            "Parquet footers"
        )

print(f"\nParquet file count: {len(parquet_files)}")
print(f"shards by split: {dict(split_shard_counts)}")
print(f"rows by split: {dict(split_row_counts)}")
print(
    "size GiB by split: "
    + str(
        {
            split: round(size / (1024**3), 3)
            for split, size in split_size_bytes.items()
        }
    )
)
print(f"schema mismatches: {schema_mismatches}")


print("\n=== VIMD SCHEMAS ===")

for split, schema in sorted(reference_schemas.items()):
    print(f"\n[{split}]")
    print(schema)


print("\n=== VIMD NON-AUDIO ROW PREVIEW ===")

first_train_path = next(
    (
        path
        for path in parquet_files
        if path.name.startswith("train-")
    ),
    parquet_files[0],
)

first_train_file = parquet.ParquetFile(first_train_path)
first_train_schema = first_train_file.schema_arrow

preview_columns = [
    field.name
    for field in first_train_schema
    if not contains_binary(field.type)
]

print(f"source shard: {first_train_path.name}")
print(f"preview columns: {preview_columns}")

preview_table = first_train_file.read_row_group(
    0,
    columns=preview_columns,
).slice(0, 3)

for index, row in enumerate(
    preview_table.to_pylist(),
    start=1,
):
    print(f"row {index}: {truncate_value(row)}")


print("\nDATASET METADATA AUDIT PASSED")

=== TIDYVOICE FILE INVENTORY ===
root: /kaggle/input/datasets/dullahn/mozzila-tidyvoice/TidyVoiceX_ASV
exists: True
total files: 321,711
files by branch: {'dev': 59443, 'train': 262268}
audio files by branch: {'dev': 59443, 'train': 262268}

file extensions:
  .wav: 321,711

manifest-like files:

representative audio paths:
  [dev]
    TidyVoiceX_Dev/TidyVoiceX_Dev/id014060/lg/lg_39353814.wav
    TidyVoiceX_Dev/TidyVoiceX_Dev/id014060/lg/lg_39353946.wav
    TidyVoiceX_Dev/TidyVoiceX_Dev/id014060/lg/lg_39353948.wav
    TidyVoiceX_Dev/TidyVoiceX_Dev/id014060/lg/lg_39353807.wav
    TidyVoiceX_Dev/TidyVoiceX_Dev/id014060/lg/lg_39354800.wav
    TidyVoiceX_Dev/TidyVoiceX_Dev/id014060/lg/lg_39354053.wav
    TidyVoiceX_Dev/TidyVoiceX_Dev/id014060/lg/lg_39187197.wav
    TidyVoiceX_Dev/TidyVoiceX_Dev/id014060/lg/lg_39353817.wav
  [train]
    TidyVoiceX_Train/TidyVoiceX_Train/id012711/fa/fa_29855668.wav
    TidyVoiceX_Train/TidyVoiceX_Train/id012711/fa/fa_29855667.wav
    TidyVoiceX_Train/TidyVoi

In [4]:
"""Audit speaker distributions and cross-split leakage without decoding audio."""

from __future__ import annotations

import os
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
from typing import Hashable

import numpy as np
import pyarrow.parquet as parquet


PUBLISHER_ROOT = Path("/kaggle/input/datasets/dullahn")

TIDYVOICE_ROOTS = {
    "train": (
        PUBLISHER_ROOT
        / "mozzila-tidyvoice"
        / "TidyVoiceX_ASV"
        / "TidyVoiceX_Train"
        / "TidyVoiceX_Train"
    ),
    "dev": (
        PUBLISHER_ROOT
        / "mozzila-tidyvoice"
        / "TidyVoiceX_ASV"
        / "TidyVoiceX_Dev"
        / "TidyVoiceX_Dev"
    ),
}

VIMD_DATA_ROOT = (
    PUBLISHER_ROOT
    / "vimd-dataset"
    / "data"
)


def summarize_counts(counter: Counter[Hashable]) -> dict[str, float]:
    """Summarize the number of utterances per categorical identity."""
    values = np.asarray(
        list(counter.values()),
        dtype=np.float64,
    )

    if values.size == 0:
        return {}

    return {
        "minimum": int(values.min()),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "mean": float(values.mean()),
        "p75": float(np.percentile(values, 75)),
        "maximum": int(values.max()),
    }


def print_pairwise_overlaps(
    values_by_split: dict[str, set[str]],
    label: str,
) -> None:
    """Print pairwise set overlaps between supplied dataset splits."""
    print(f"\n{label} overlaps:")

    for first_split, second_split in combinations(
        sorted(values_by_split),
        2,
    ):
        overlap = (
            values_by_split[first_split]
            & values_by_split[second_split]
        )

        print(
            f"  {first_split} vs {second_split}: "
            f"{len(overlap):,}"
        )

        if overlap:
            print(
                "    examples: "
                f"{sorted(overlap)[:10]}"
            )


print("=== TIDYVOICE PATH AND SPEAKER AUDIT ===")

tidy_speakers: dict[str, set[str]] = {}
tidy_utterances_per_speaker: dict[str, Counter[str]] = {}
tidy_language_counts: dict[str, Counter[str]] = {}
tidy_speakers_per_language: dict[str, dict[str, set[str]]] = {}
tidy_speaker_languages: dict[str, dict[str, set[str]]] = {}
tidy_invalid_paths: dict[str, list[str]] = {}

for split, split_root in TIDYVOICE_ROOTS.items():
    print(f"\n[{split}] root: {split_root}")
    assert split_root.is_dir()

    speakers: set[str] = set()
    utterances_per_speaker: Counter[str] = Counter()
    language_counts: Counter[str] = Counter()
    speakers_per_language: dict[str, set[str]] = defaultdict(set)
    speaker_languages: dict[str, set[str]] = defaultdict(set)
    invalid_paths: list[str] = []
    total_files = 0

    for directory_path, _, filenames in os.walk(split_root):
        directory = Path(directory_path)

        for filename in filenames:
            file_path = directory / filename

            if file_path.suffix.lower() != ".wav":
                continue

            relative_path = file_path.relative_to(split_root)
            path_parts = relative_path.parts
            total_files += 1

            # Expected layout: speaker/language/utterance.wav.
            if len(path_parts) != 3:
                if len(invalid_paths) < 20:
                    invalid_paths.append(str(relative_path))
                continue

            speaker_id, language, _ = path_parts

            speakers.add(speaker_id)
            utterances_per_speaker[speaker_id] += 1
            language_counts[language] += 1
            speakers_per_language[language].add(speaker_id)
            speaker_languages[speaker_id].add(language)

    tidy_speakers[split] = speakers
    tidy_utterances_per_speaker[split] = utterances_per_speaker
    tidy_language_counts[split] = language_counts
    tidy_speakers_per_language[split] = speakers_per_language
    tidy_speaker_languages[split] = speaker_languages
    tidy_invalid_paths[split] = invalid_paths

    multilingual_speaker_count = sum(
        len(languages) > 1
        for languages in speaker_languages.values()
    )

    print(f"  WAV files: {total_files:,}")
    print(f"  valid speaker paths: {sum(utterances_per_speaker.values()):,}")
    print(f"  invalid path count: {total_files - sum(utterances_per_speaker.values()):,}")
    print(f"  speakers: {len(speakers):,}")
    print(f"  languages: {len(language_counts):,}")
    print(
        "  utterances per speaker: "
        f"{summarize_counts(utterances_per_speaker)}"
    )
    print(
        "  speakers with multiple language folders: "
        f"{multilingual_speaker_count:,}"
    )

    print("  largest language groups:")
    for language, count in language_counts.most_common(20):
        print(
            f"    {language}: "
            f"{count:,} utterances, "
            f"{len(speakers_per_language[language]):,} speakers"
        )

    if invalid_paths:
        print(f"  invalid path examples: {invalid_paths}")

print_pairwise_overlaps(
    tidy_speakers,
    "TidyVoice speaker",
)


print("\n=== VIMD SPEAKER AND LABEL AUDIT ===")

vimd_columns = [
    "region",
    "province_code",
    "province_name",
    "filename",
    "speakerID",
    "gender",
]

vimd_speakers: dict[str, set[str]] = defaultdict(set)
vimd_utterances_per_speaker: dict[str, Counter[str]] = defaultdict(Counter)
vimd_regions: dict[str, Counter[str]] = defaultdict(Counter)
vimd_provinces: dict[str, Counter[str]] = defaultdict(Counter)
vimd_genders: dict[str, Counter[int]] = defaultdict(Counter)
vimd_record_keys: dict[str, Counter[tuple[str, str]]] = defaultdict(Counter)
vimd_empty_values: dict[str, Counter[str]] = defaultdict(Counter)

speaker_regions: dict[str, set[str]] = defaultdict(set)
speaker_provinces: dict[str, set[str]] = defaultdict(set)
speaker_genders: dict[str, set[int]] = defaultdict(set)

parquet_files = sorted(VIMD_DATA_ROOT.glob("*.parquet"))
assert parquet_files

for index, parquet_path in enumerate(parquet_files, start=1):
    split = parquet_path.name.split("-", maxsplit=1)[0]

    table = parquet.ParquetFile(parquet_path).read(
        columns=vimd_columns
    )
    columns = table.to_pydict()

    for (
        region,
        province_code,
        province_name,
        filename,
        speaker_id,
        gender,
    ) in zip(
        columns["region"],
        columns["province_code"],
        columns["province_name"],
        columns["filename"],
        columns["speakerID"],
        columns["gender"],
        strict=True,
    ):
        field_values = {
            "region": region,
            "province_code": province_code,
            "province_name": province_name,
            "filename": filename,
            "speakerID": speaker_id,
            "gender": gender,
        }

        for field_name, value in field_values.items():
            if value is None or value == "":
                vimd_empty_values[split][field_name] += 1

        # Keep processing defensive: missing speaker IDs are counted above
        # but excluded from identity statistics.
        if not speaker_id:
            continue

        normalized_region = str(region)
        normalized_province = str(province_name)
        normalized_filename = str(filename)
        normalized_gender = int(gender)

        vimd_speakers[split].add(speaker_id)
        vimd_utterances_per_speaker[split][speaker_id] += 1
        vimd_regions[split][normalized_region] += 1
        vimd_provinces[split][normalized_province] += 1
        vimd_genders[split][normalized_gender] += 1
        vimd_record_keys[split][
            (speaker_id, normalized_filename)
        ] += 1

        speaker_regions[speaker_id].add(normalized_region)
        speaker_provinces[speaker_id].add(normalized_province)
        speaker_genders[speaker_id].add(normalized_gender)

    if index % 25 == 0 or index == len(parquet_files):
        print(
            f"processed {index}/{len(parquet_files)} "
            "metadata shards"
        )

for split in sorted(vimd_speakers):
    duplicate_record_count = sum(
        count - 1
        for count in vimd_record_keys[split].values()
        if count > 1
    )

    print(f"\n[{split}]")
    print(
        f"  utterances: "
        f"{sum(vimd_utterances_per_speaker[split].values()):,}"
    )
    print(f"  speakers: {len(vimd_speakers[split]):,}")
    print(
        "  utterances per speaker: "
        f"{summarize_counts(vimd_utterances_per_speaker[split])}"
    )
    print(f"  regions: {dict(vimd_regions[split])}")
    print(f"  genders: {dict(vimd_genders[split])}")
    print(f"  duplicate speaker/filename keys: {duplicate_record_count:,}")
    print(f"  empty values: {dict(vimd_empty_values[split])}")

    print("  largest province groups:")
    for province, count in vimd_provinces[split].most_common(15):
        print(f"    {province}: {count:,}")

print_pairwise_overlaps(
    dict(vimd_speakers),
    "ViMD speaker",
)

print(
    "\nViMD speakers associated with multiple regions: "
    f"{sum(len(values) > 1 for values in speaker_regions.values()):,}"
)
print(
    "ViMD speakers associated with multiple provinces: "
    f"{sum(len(values) > 1 for values in speaker_provinces.values()):,}"
)
print(
    "ViMD speakers associated with multiple genders: "
    f"{sum(len(values) > 1 for values in speaker_genders.values()):,}"
)

print("\nSPEAKER AND SPLIT-INTEGRITY AUDIT PASSED")

=== TIDYVOICE PATH AND SPEAKER AUDIT ===

[train] root: /kaggle/input/datasets/dullahn/mozzila-tidyvoice/TidyVoiceX_ASV/TidyVoiceX_Train/TidyVoiceX_Train
  WAV files: 262,268
  valid speaker paths: 262,268
  invalid path count: 0
  speakers: 3,666
  languages: 40
  utterances per speaker: {'minimum': 4, 'p25': 14.0, 'median': 28.0, 'mean': 71.54064375340971, 'p75': 76.0, 'maximum': 1618}
  speakers with multiple language folders: 3,666
  largest language groups:
    en: 78,596 utterances, 3,297 speakers
    de: 55,844 utterances, 1,070 speakers
    fr: 31,375 utterances, 814 speakers
    ru: 13,379 utterances, 394 speakers
    nl: 12,211 utterances, 226 speakers
    be: 10,477 utterances, 171 speakers
    fa: 7,330 utterances, 137 speakers
    pt: 6,685 utterances, 166 speakers
    ca: 6,160 utterances, 77 speakers
    pl: 5,963 utterances, 138 speakers
    zh-CN: 4,994 utterances, 146 speakers
    cy: 3,639 utterances, 69 speakers
    ta: 3,193 utterances, 76 speakers
    ar: 2,930 ut

In [5]:
"""Measure speaker-verification feasibility from existing audit counters."""

from __future__ import annotations

from collections import Counter
from math import comb


def report_verification_capacity(
    dataset: str,
    split: str,
    utterances_per_speaker: Counter[str],
) -> None:
    """Report speaker eligibility and theoretical genuine-pair capacity."""
    print(f"\n[{dataset} / {split}]")

    total_speakers = len(utterances_per_speaker)
    total_utterances = sum(utterances_per_speaker.values())

    print(f"speakers: {total_speakers:,}")
    print(f"utterances: {total_utterances:,}")

    for minimum_utterances in (2, 3, 4, 5, 10):
        eligible_counts = [
            count
            for count in utterances_per_speaker.values()
            if count >= minimum_utterances
        ]

        retained_utterances = sum(eligible_counts)
        retained_percentage = (
            100 * retained_utterances / total_utterances
            if total_utterances
            else 0
        )

        print(
            f"speakers with >= {minimum_utterances} utterances: "
            f"{len(eligible_counts):,}; "
            f"retained utterances: {retained_utterances:,} "
            f"({retained_percentage:.2f}%)"
        )

    genuine_pair_capacity = sum(
        comb(count, 2)
        for count in utterances_per_speaker.values()
        if count >= 2
    )

    print(
        "theoretical unique genuine pairs: "
        f"{genuine_pair_capacity:,}"
    )


assert "tidy_utterances_per_speaker" in globals()
assert "vimd_utterances_per_speaker" in globals()

print("=== VERIFICATION FEASIBILITY ===")

for split, counter in sorted(
    tidy_utterances_per_speaker.items()
):
    report_verification_capacity(
        "TidyVoice",
        split,
        counter,
    )

for split, counter in sorted(
    vimd_utterances_per_speaker.items()
):
    report_verification_capacity(
        "ViMD",
        split,
        counter,
    )

print("\nVERIFICATION FEASIBILITY AUDIT PASSED")

=== VERIFICATION FEASIBILITY ===

[TidyVoice / dev]
speakers: 808
utterances: 59,443
speakers with >= 2 utterances: 808; retained utterances: 59,443 (100.00%)
speakers with >= 3 utterances: 808; retained utterances: 59,443 (100.00%)
speakers with >= 4 utterances: 808; retained utterances: 59,443 (100.00%)
speakers with >= 5 utterances: 801; retained utterances: 59,415 (99.95%)
speakers with >= 10 utterances: 677; retained utterances: 58,480 (98.38%)
theoretical unique genuine pairs: 7,405,777

[TidyVoice / train]
speakers: 3,666
utterances: 262,268
speakers with >= 2 utterances: 3,666; retained utterances: 262,268 (100.00%)
speakers with >= 3 utterances: 3,666; retained utterances: 262,268 (100.00%)
speakers with >= 4 utterances: 3,666; retained utterances: 262,268 (100.00%)
speakers with >= 5 utterances: 3,647; retained utterances: 262,192 (99.97%)
speakers with >= 10 utterances: 3,163; retained utterances: 258,564 (98.59%)
theoretical unique genuine pairs: 31,829,370

[ViMD / test]
s

In [6]:
"""Perform a deterministic, memory-safe audio-header audit."""

from __future__ import annotations

import io
import os
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import BinaryIO

import numpy as np
import pyarrow.parquet as parquet
import soundfile


ROOT = Path("/kaggle/input/datasets/dullahn")

TIDY_ROOTS = {
    "train": (
        ROOT / "mozzila-tidyvoice/TidyVoiceX_ASV/"
        "TidyVoiceX_Train/TidyVoiceX_Train"
    ),
    "dev": (
        ROOT / "mozzila-tidyvoice/TidyVoiceX_ASV/"
        "TidyVoiceX_Dev/TidyVoiceX_Dev"
    ),
}

VIMD_ROOT = ROOT / "vimd-dataset/data"
SEED = 42


def reservoir_sample_wavs(
    root: Path,
    sample_size: int,
    seed: int,
) -> list[Path]:
    """Select WAV files uniformly without retaining every path."""
    generator = random.Random(seed)
    sample: list[Path] = []
    seen = 0

    for directory, subdirectories, filenames in os.walk(root):
        subdirectories.sort()

        for filename in sorted(filenames):
            if not filename.lower().endswith(".wav"):
                continue

            path = Path(directory) / filename
            seen += 1

            if len(sample) < sample_size:
                sample.append(path)
                continue

            replacement_index = generator.randrange(seen)

            if replacement_index < sample_size:
                sample[replacement_index] = path

    return sample


def summarize_headers(
    label: str,
    records: list[dict[str, object]],
    errors: list[str],
) -> None:
    """Print distributions from collected audio-header records."""
    print(f"\n[{label}]")
    print(f"inspected files: {len(records):,}")
    print(f"header errors: {len(errors):,}")

    if errors:
        print(f"error examples: {errors[:10]}")

    if not records:
        return

    durations = np.asarray(
        [record["duration"] for record in records],
        dtype=np.float64,
    )

    print(
        "sample rates: "
        f"{dict(Counter(record['sample_rate'] for record in records))}"
    )
    print(
        "channels: "
        f"{dict(Counter(record['channels'] for record in records))}"
    )
    print(
        "formats: "
        f"{dict(Counter(record['format'] for record in records))}"
    )
    print(
        "subtypes: "
        f"{dict(Counter(record['subtype'] for record in records))}"
    )
    print(
        "duration seconds: "
        f"min={durations.min():.3f}, "
        f"p05={np.percentile(durations, 5):.3f}, "
        f"median={np.median(durations):.3f}, "
        f"p95={np.percentile(durations, 95):.3f}, "
        f"max={durations.max():.3f}"
    )


def inspect_audio(
    source: str | Path | BinaryIO,
    identifier: str,
) -> dict[str, object]:
    """Read only an audio header and return normalized metadata."""
    information = soundfile.info(source)

    return {
        "identifier": identifier,
        "sample_rate": information.samplerate,
        "channels": information.channels,
        "frames": information.frames,
        "duration": information.frames / information.samplerate,
        "format": information.format,
        "subtype": information.subtype,
    }


print("=== TIDYVOICE HEADER SAMPLE ===")

tidy_records: dict[str, list[dict[str, object]]] = defaultdict(list)
tidy_errors: dict[str, list[str]] = defaultdict(list)

for split, root in TIDY_ROOTS.items():
    sample_size = 256 if split == "train" else 128
    sampled_paths = reservoir_sample_wavs(
        root,
        sample_size,
        SEED,
    )

    for path in sampled_paths:
        try:
            tidy_records[split].append(
                inspect_audio(
                    path,
                    str(path.relative_to(root)),
                )
            )
        except Exception as error:
            tidy_errors[split].append(
                f"{path}: {type(error).__name__}: {error}"
            )

    summarize_headers(
        f"TidyVoice {split}",
        tidy_records[split],
        tidy_errors[split],
    )


print("\n=== VIMD HEADER SAMPLE ===")

vimd_records: dict[str, list[dict[str, object]]] = defaultdict(list)
vimd_errors: dict[str, list[str]] = defaultdict(list)

for split in ("train", "valid", "test"):
    split_files = sorted(VIMD_ROOT.glob(f"{split}-*.parquet"))
    assert split_files

    # Sample five shards spread across the entire split.
    shard_indices = np.linspace(
        0,
        len(split_files) - 1,
        num=min(5, len(split_files)),
        dtype=int,
    )

    selected_shards = [
        split_files[index]
        for index in sorted(set(shard_indices.tolist()))
    ]

    for shard_path in selected_shards:
        batches = parquet.ParquetFile(shard_path).iter_batches(
            batch_size=1,
            columns=["filename", "speakerID", "audio"],
        )

        # Inspect four rows per selected shard while keeping memory bounded.
        for row_index, batch in enumerate(batches):
            if row_index >= 4:
                break

            row = batch.to_pylist()[0]
            audio_bytes = row["audio"]["bytes"]
            identifier = (
                f"{shard_path.name}:"
                f"{row['speakerID']}/{row['filename']}"
            )

            try:
                vimd_records[split].append(
                    inspect_audio(
                        io.BytesIO(audio_bytes),
                        identifier,
                    )
                )
            except Exception as error:
                vimd_errors[split].append(
                    f"{identifier}: "
                    f"{type(error).__name__}: {error}"
                )

    summarize_headers(
        f"ViMD {split}",
        vimd_records[split],
        vimd_errors[split],
    )

assert all(tidy_records.values())
assert all(vimd_records.values())

print("\nAUDIO HEADER AUDIT PASSED")

=== TIDYVOICE HEADER SAMPLE ===

[TidyVoice train]
inspected files: 256
header errors: 0
sample rates: {16000: 256}
channels: {1: 256}
formats: {'WAV': 256}
subtypes: {'PCM_16': 256}
duration seconds: min=1.368, p05=2.367, median=5.040, p95=7.983, max=9.540

[TidyVoice dev]
inspected files: 128
header errors: 0
sample rates: {16000: 128}
channels: {1: 128}
formats: {'WAV': 128}
subtypes: {'PCM_16': 128}
duration seconds: min=1.836, p05=2.675, median=5.232, p95=8.145, max=9.552

=== VIMD HEADER SAMPLE ===

[ViMD train]
inspected files: 20
header errors: 0
sample rates: {44100: 20}
channels: {2: 18, 1: 2}
formats: {'WAV': 20}
subtypes: {'PCM_16': 20}
duration seconds: min=10.411, p05=13.233, median=21.634, p95=29.247, max=31.288

[ViMD valid]
inspected files: 20
header errors: 0
sample rates: {44100: 20}
channels: {2: 20}
formats: {'WAV': 20}
subtypes: {'PCM_16': 20}
duration seconds: min=7.043, p05=11.910, median=21.478, p95=29.662, max=29.692

[ViMD test]
inspected files: 20
header err

In [7]:
"""Simulate a deterministic speaker-disjoint TidyVoice validation/test split."""

from __future__ import annotations

import os
import random
from collections import Counter, defaultdict
from pathlib import Path


DEV_ROOT = Path(
    "/kaggle/input/datasets/dullahn/mozzila-tidyvoice/"
    "TidyVoiceX_ASV/TidyVoiceX_Dev/TidyVoiceX_Dev"
)
SEED = 42


speaker_language_counts: dict[str, Counter[str]] = defaultdict(Counter)
speaker_utterance_counts: Counter[str] = Counter()

for directory, subdirectories, filenames in os.walk(DEV_ROOT):
    subdirectories.sort()
    relative_directory = Path(directory).relative_to(DEV_ROOT)

    if len(relative_directory.parts) != 2:
        continue

    speaker_id, language = relative_directory.parts

    for filename in sorted(filenames):
        if filename.lower().endswith(".wav"):
            speaker_language_counts[speaker_id][language] += 1
            speaker_utterance_counts[speaker_id] += 1


speaker_ids = sorted(speaker_utterance_counts)
generator = random.Random(SEED)
generator.shuffle(speaker_ids)

split_index = len(speaker_ids) // 2
validation_speakers = set(speaker_ids[:split_index])
test_speakers = set(speaker_ids[split_index:])


def summarize_partition(
    name: str,
    speakers: set[str],
) -> tuple[int, Counter[str]]:
    """Print utterance and language statistics for one partition."""
    language_counts: Counter[str] = Counter()

    for speaker_id in speakers:
        language_counts.update(
            speaker_language_counts[speaker_id]
        )

    utterance_count = sum(language_counts.values())

    print(f"\n[{name}]")
    print(f"speakers: {len(speakers):,}")
    print(f"utterances: {utterance_count:,}")
    print("largest languages:")

    for language, count in language_counts.most_common(15):
        percentage = 100 * count / utterance_count
        print(
            f"  {language}: {count:,} "
            f"({percentage:.2f}%)"
        )

    return utterance_count, language_counts


print("=== TIDYVOICE DEV SPLIT CANDIDATE ===")
print(f"seed: {SEED}")
print(f"speaker overlap: {len(validation_speakers & test_speakers)}")

validation_count, validation_languages = summarize_partition(
    "validation",
    validation_speakers,
)
test_count, test_languages = summarize_partition(
    "test",
    test_speakers,
)

total_count = validation_count + test_count
utterance_imbalance = (
    abs(validation_count - test_count) / total_count
)

all_languages = (
    set(validation_languages)
    | set(test_languages)
)

maximum_proportion_difference = max(
    abs(
        validation_languages[language] / validation_count
        - test_languages[language] / test_count
    )
    for language in all_languages
)

print(
    "\nutterance imbalance: "
    f"{100 * utterance_imbalance:.3f}%"
)
print(
    "maximum language-proportion difference: "
    f"{100 * maximum_proportion_difference:.3f} percentage points"
)

assert not validation_speakers & test_speakers
assert validation_speakers | test_speakers == set(speaker_ids)

print("\nTIDYVOICE DEV SPLIT SIMULATION PASSED")

=== TIDYVOICE DEV SPLIT CANDIDATE ===
seed: 42
speaker overlap: 0

[validation]
speakers: 404
utterances: 26,340
largest languages:
  en: 5,680 (21.56%)
  ba: 1,602 (6.08%)
  ru: 1,265 (4.80%)
  be: 1,229 (4.67%)
  de: 1,102 (4.18%)
  lg: 1,099 (4.17%)
  nl: 1,070 (4.06%)
  uz: 1,069 (4.06%)
  fa: 1,010 (3.83%)
  cy: 989 (3.75%)
  dv: 903 (3.43%)
  lt: 852 (3.23%)
  zh-CN: 829 (3.15%)
  ar: 774 (2.94%)
  yue: 711 (2.70%)

[test]
speakers: 404
utterances: 33,103
largest languages:
  en: 7,859 (23.74%)
  de: 2,041 (6.17%)
  ca: 1,841 (5.56%)
  fr: 1,618 (4.89%)
  be: 1,371 (4.14%)
  pl: 1,364 (4.12%)
  ka: 1,279 (3.86%)
  tr: 1,233 (3.72%)
  ar: 1,056 (3.19%)
  lg: 1,028 (3.11%)
  hi: 923 (2.79%)
  ug: 883 (2.67%)
  nl: 871 (2.63%)
  mr: 645 (1.95%)
  ta: 644 (1.95%)

utterance imbalance: 11.377%
maximum language-proportion difference: 4.538 percentage points

TIDYVOICE DEV SPLIT SIMULATION PASSED


## Reproducible TidyVoice Protocol Generation

In [1]:
from pathlib import Path

repository_path = Path("/kaggle/working/Who-Speak-AI")
print("already exists:", repository_path.exists())

already exists: False


In [2]:
!git clone --depth 1 --branch thanhDT "https://github.com/beaver-felix/Who-Speak-AI.git" /kaggle/working/Who-Speak-AI

Cloning into '/kaggle/working/Who-Speak-AI'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 45 (delta 4), reused 32 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 57.01 KiB | 14.25 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [3]:
%cd /kaggle/working/Who-Speak-AI/model/Thanh2
!python -m pip install -e ".[dev]"

/kaggle/working/Who-Speak-AI/model/Thanh2
Obtaining file:///kaggle/working/Who-Speak-AI/model/Thanh2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for who-speak-ai (pyproject.toml) ... done
  Created wheel for who-speak-ai: filename=who_speak_ai-0.1.0-0.editable-py3-none-any.whl size=1365 sha256=94a168e6d2fcaafab48a0d4e2f09de4f2675eddf3fc72068e639c022c6b7de13
  Stored in directory: /tmp/pip-ephem-wheel-cache-4iqxv8hx/wheels/dd/37/19/34f43b94ad38dea9ffe1521f4c327b33de3a8b281f533a1285
Successfully built who-speak-ai


In [4]:
!python -m pytest -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/Who-Speak-AI/model/Thanh2
configfile: pyproject.toml
testpaths: tests
plugins: anyio-4.13.0, typeguard-4.5.1, langsmith-0.7.34
collected 26 items                                                             

tests/data/test_manifest.py::test_file_record_normalizes_path_separators PASSED [  3%]
tests/data/test_manifest.py::test_parquet_record_requires_row_index PASSED [  7%]
tests/data/test_manifest.py::test_manifest_accepts_disjoint_groups PASSED [ 11%]
tests/data/test_manifest.py::test_manifest_rejects_duplicate_utterance_id PASSED [ 15%]
tests/data/test_manifest.py::test_manifest_rejects_duplicate_physical_audio PASSED [ 19%]
tests/data/test_manifest.py::test_manifest_rejects_speaker_leakage PASSED [ 23%]
tests/data/test_manifest.py::test_manifest_rejects_recording_leakage PAS

In [5]:
!python scripts/prepare_tidyvoice_protocol.py \
    --dataset-root "/kaggle/input/datasets/dullahn/mozzila-tidyvoice/TidyVoiceX_ASV" \
    --output "results/data_audit/tidyvoice_dev_protocol.json" \
    --seed 42 \
    --validation-fraction 0.5 \
    --restarts 64 \
    --max-swap-passes 8

TIDYVOICE DEV PROTOCOL GENERATED
speakers: 808
utterances: 59,443
validation/test speakers: 404/404
validation/test utterances: 29,720/29,723
item imbalance: 0.005047%
maximum language-proportion difference: 2.230175%
objective: 0.02235222
profiles SHA-256: 9e5c0b2502f732307a605e6ccdc7dda763cd1903badfea95ef7922bcb4800b9a
output: /kaggle/working/Who-Speak-AI/model/Thanh2/results/data_audit/tidyvoice_dev_protocol.json


In [6]:
import json
from pathlib import Path

protocol_path = Path(
    "/kaggle/working/Who-Speak-AI/model/Thanh2/"
    "results/data_audit/tidyvoice_dev_protocol.json"
)
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))

assert protocol["schema_version"] == 1
assert len(protocol["assignments"]) == 808
assert set(protocol["assignments"].values()) == {"validation", "test"}

print("artifact size:", protocol_path.stat().st_size, "bytes")
print("PROTOCOL ARTIFACT VALIDATION PASSED")

artifact size: 30744 bytes
PROTOCOL ARTIFACT VALIDATION PASSED


In [7]:
from IPython.display import FileLink

FileLink(
    "/kaggle/working/Who-Speak-AI/model/Thanh2/"
    "results/data_audit/tidyvoice_dev_protocol.json"
)

/kaggle/working/Who-Speak-AI/model/Thanh2/results/data_audit/tidyvoice_dev_protocol.json

In [8]:
from pathlib import Path
from shutil import copy2

source = Path(
    "/kaggle/working/Who-Speak-AI/model/Thanh2/"
    "results/data_audit/tidyvoice_dev_protocol.json"
)
destination = Path("/kaggle/working/tidyvoice_dev_protocol.json")

copy2(source, destination)
print(destination, destination.stat().st_size)

/kaggle/working/tidyvoice_dev_protocol.json 30744


### Selected TidyVoice Dev Protocol

The deterministic metadata-balanced protocol produced 404 validation and 404
test speakers with no speaker overlap. It reduced utterance imbalance from
11.377% to 0.005047% and reduced the maximum language-proportion difference
from 4.538 to 2.230175 percentage points.

The accepted assignments are stored in
`results/data_audit/tidyvoice_dev_protocol.json`.

## ViMD Canonical Protocol Resolution

In [10]:
from collections import Counter
from pathlib import Path

import pyarrow.parquet as pq

vimd_root = Path("/kaggle/input/datasets/dullahn/vimd-dataset/data")
overlapping_speakers = {"spk_73_0186", "spk_76_0219"}
selected_columns = [
    "speakerID",
    "filename",
    "region",
    "province_name",
    "gender",
]

overlap_rows = []

for source_split in ("valid", "test"):
    shard_paths = sorted(vimd_root.glob(f"{source_split}-*.parquet"))

    for shard_path in shard_paths:
        parquet_file = pq.ParquetFile(shard_path)
        row_offset = 0

        for batch in parquet_file.iter_batches(
            batch_size=4096,
            columns=selected_columns,
        ):
            columns = batch.to_pydict()

            for batch_index, speaker_id in enumerate(columns["speakerID"]):
                if speaker_id not in overlapping_speakers:
                    continue

                overlap_rows.append(
                    {
                        "speakerID": speaker_id,
                        "source_split": source_split,
                        "filename": columns["filename"][batch_index],
                        "region": columns["region"][batch_index],
                        "province_name": columns["province_name"][batch_index],
                        "gender": columns["gender"][batch_index],
                        "shard": shard_path.name,
                        "row_index": row_offset + batch_index,
                    }
                )

            row_offset += batch.num_rows

overlap_rows.sort(
    key=lambda row: (
        row["speakerID"],
        row["source_split"],
        row["filename"],
    )
)

print("=== OVERLAPPING SPEAKER ROWS ===")
for row in overlap_rows:
    print(row)

print("\n=== COUNTS BY SPEAKER AND SOURCE SPLIT ===")
counts = Counter(
    (row["speakerID"], row["source_split"])
    for row in overlap_rows
)
for key in sorted(counts):
    print(f"{key}: {counts[key]}")

assert {row["speakerID"] for row in overlap_rows} == overlapping_speakers
print("\nVIMD OVERLAP DETAIL AUDIT PASSED")

=== OVERLAPPING SPEAKER ROWS ===
{'speakerID': 'spk_73_0186', 'source_split': 'test', 'filename': '73_0307.wav', 'region': 'Central', 'province_name': 'QuangBinh', 'gender': 0, 'shard': 'test-00008-of-00014.parquet', 'row_index': 134}
{'speakerID': 'spk_73_0186', 'source_split': 'test', 'filename': '73_0308.wav', 'region': 'Central', 'province_name': 'QuangBinh', 'gender': 0, 'shard': 'test-00008-of-00014.parquet', 'row_index': 135}
{'speakerID': 'spk_73_0186', 'source_split': 'valid', 'filename': '73_0309.wav', 'region': 'Central', 'province_name': 'QuangBinh', 'gender': 1, 'shard': 'valid-00008-of-00013.parquet', 'row_index': 89}
{'speakerID': 'spk_76_0219', 'source_split': 'test', 'filename': '76_0294.wav', 'region': 'Central', 'province_name': 'QuangNgai', 'gender': 1, 'shard': 'test-00009-of-00014.parquet', 'row_index': 92}
{'speakerID': 'spk_76_0219', 'source_split': 'valid', 'filename': '76_0295.wav', 'region': 'Central', 'province_name': 'QuangNgai', 'gender': 0, 'shard': 'vali

### Candidate Policy: Preserve Test and Exclude Contaminated Validation Rows

The official ViMD Test partition remains unchanged. Validation utterances whose
speaker identity occurs in Test are excluded from the canonical protocol. This
avoids speaker leakage without moving validation data into the final test set.

In [11]:
from collections import Counter
from pathlib import Path

import pyarrow.parquet as pq

vimd_root = Path("/kaggle/input/datasets/dullahn/vimd-dataset/data")


def collect_vimd_speaker_counts(
    source_split: str,
) -> Counter[str]:
    """Count ViMD utterances per speaker without reading audio bytes."""
    counts: Counter[str] = Counter()

    for shard_path in sorted(vimd_root.glob(f"{source_split}-*.parquet")):
        parquet_file = pq.ParquetFile(shard_path)

        for batch in parquet_file.iter_batches(
            batch_size=4096,
            columns=["speakerID"],
        ):
            counts.update(batch.column("speakerID").to_pylist())

    return counts


def count_genuine_pairs(
    speaker_counts: Counter[str],
) -> int:
    """Count all unique same-speaker utterance pairs."""
    return sum(
        utterance_count * (utterance_count - 1) // 2
        for utterance_count in speaker_counts.values()
    )


source_counts = {
    split: collect_vimd_speaker_counts(split)
    for split in ("train", "valid", "test")
}

excluded_validation_speakers = (
    set(source_counts["valid"]) & set(source_counts["test"])
)
canonical_validation_counts = Counter(
    {
        speaker_id: count
        for speaker_id, count in source_counts["valid"].items()
        if speaker_id not in excluded_validation_speakers
    }
)

canonical_speaker_sets = {
    "train": set(source_counts["train"]),
    "validation": set(canonical_validation_counts),
    "test": set(source_counts["test"]),
}

excluded_validation_utterances = sum(
    source_counts["valid"][speaker_id]
    for speaker_id in excluded_validation_speakers
)

print("=== CANONICAL VIMD POLICY ===")
print(
    "excluded validation speakers:",
    sorted(excluded_validation_speakers),
)
print(
    "excluded validation utterances:",
    excluded_validation_utterances,
)

print("\n=== CANONICAL COUNTS ===")
print(
    "train:",
    sum(source_counts["train"].values()),
    "utterances,",
    len(source_counts["train"]),
    "speakers",
)
print(
    "validation:",
    sum(canonical_validation_counts.values()),
    "utterances,",
    len(canonical_validation_counts),
    "speakers",
)
print(
    "test:",
    sum(source_counts["test"].values()),
    "utterances,",
    len(source_counts["test"]),
    "speakers",
)

print("\n=== GENUINE-PAIR CAPACITY ===")
print("source validation:", count_genuine_pairs(source_counts["valid"]))
print(
    "canonical validation:",
    count_genuine_pairs(canonical_validation_counts),
)
print("test:", count_genuine_pairs(source_counts["test"]))

assert excluded_validation_speakers == {
    "spk_73_0186",
    "spk_76_0219",
}
assert excluded_validation_utterances == 2
assert not (
    canonical_speaker_sets["train"]
    & canonical_speaker_sets["validation"]
)
assert not (
    canonical_speaker_sets["train"]
    & canonical_speaker_sets["test"]
)
assert not (
    canonical_speaker_sets["validation"]
    & canonical_speaker_sets["test"]
)

print("\nVIMD CANONICAL SPEAKER-DISJOINT POLICY PASSED")

=== CANONICAL VIMD POLICY ===
excluded validation speakers: ['spk_73_0186', 'spk_76_0219']
excluded validation utterances: 2

=== CANONICAL COUNTS ===
train: 15023 utterances, 10291 speakers
validation: 1898 utterances, 1318 speakers
test: 2026 utterances, 1344 speakers

=== GENUINE-PAIR CAPACITY ===
source validation: 879
canonical validation: 879
test: 1046

VIMD CANONICAL SPEAKER-DISJOINT POLICY PASSED


In [12]:
import importlib.metadata as metadata

for package in ("pyarrow", "soundfile"):
    print(f"{package}: {metadata.version(package)}")

pyarrow: 24.0.0
soundfile: 0.13.1


In [16]:
!git -C /kaggle/working/Who-Speak-AI pull --ff-only origin thanhDT

From https://github.com/beaver-felix/Who-Speak-AI
 * branch            thanhDT    -> FETCH_HEAD
Updating 78c9448..8edecb5
Fast-forward
 model/Thanh2/PROJECT_CONTRACT.md                   |  18 +
 .../docs/decisions/001_tidyvoice_dev_protocol.md   |  86 +++
 model/Thanh2/notebooks/01_dataset_audit_eda.ipynb  |   1 +
 model/Thanh2/pyproject.toml                        |   4 +
 .../results/data_audit/tidyvoice_dev_protocol.json | 838 +++++++++++++++++++++
 model/Thanh2/scripts/validate_vimd_protocol.py     | 199 +++++
 model/Thanh2/src/speaker_recognition/data/vimd.py  | 259 +++++++
 model/Thanh2/tests/data/test_vimd.py               | 246 ++++++
 8 files changed, 1651 insertions(+)
 create mode 100644 model/Thanh2/docs/decisions/001_tidyvoice_dev_protocol.md
 create mode 100644 model/Thanh2/notebooks/01_dataset_audit_eda.ipynb
 create mode 100644 model/Thanh2/results/data_audit/tidyvoice_dev_protocol.json
 create mode 100644 model/Thanh2/scripts/validate_vimd_protocol.py
 create mode 100

In [14]:
import hashlib
from pathlib import Path

paths = (
    Path(
        "/kaggle/working/Who-Speak-AI/model/Thanh2/"
        "results/data_audit/tidyvoice_dev_protocol.json"
    ),
    Path("/kaggle/working/tidyvoice_dev_protocol.json"),
)

for path in paths:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(path, path.stat().st_size, digest)

/kaggle/working/Who-Speak-AI/model/Thanh2/results/data_audit/tidyvoice_dev_protocol.json 30744 9761cd365e0775c0b717b4a6df842d65975e38f81ec841d5997b8ef4a1ae3656
/kaggle/working/tidyvoice_dev_protocol.json 30744 9761cd365e0775c0b717b4a6df842d65975e38f81ec841d5997b8ef4a1ae3656


In [15]:
from pathlib import Path
from shutil import move

source = Path(
    "/kaggle/working/Who-Speak-AI/model/Thanh2/"
    "results/data_audit/tidyvoice_dev_protocol.json"
)
backup = Path(
    "/kaggle/working/tidyvoice_dev_protocol_before_pull.json"
)

assert source.is_file()
assert not backup.exists()
move(source, backup)
print("backup:", backup, backup.stat().st_size)

backup: /kaggle/working/tidyvoice_dev_protocol_before_pull.json 30744


In [17]:
from pathlib import Path
import hashlib

committed = Path(
    "/kaggle/working/Who-Speak-AI/model/Thanh2/"
    "results/data_audit/tidyvoice_dev_protocol.json"
)
backup = Path(
    "/kaggle/working/tidyvoice_dev_protocol_before_pull.json"
)

committed_hash = hashlib.sha256(committed.read_bytes()).hexdigest()
backup_hash = hashlib.sha256(backup.read_bytes()).hexdigest()

print("committed:", committed_hash)
print("generated:", backup_hash)
assert committed_hash == backup_hash
print("TIDYVOICE ARTIFACT REPRODUCIBILITY PASSED")

committed: 9761cd365e0775c0b717b4a6df842d65975e38f81ec841d5997b8ef4a1ae3656
generated: 9761cd365e0775c0b717b4a6df842d65975e38f81ec841d5997b8ef4a1ae3656
TIDYVOICE ARTIFACT REPRODUCIBILITY PASSED


In [18]:
%cd /kaggle/working/Who-Speak-AI/model/Thanh2
!python -m pip install -e ".[data]"

/kaggle/working/Who-Speak-AI/model/Thanh2
Obtaining file:///kaggle/working/Who-Speak-AI/model/Thanh2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for who-speak-ai (pyproject.toml) ... done
  Created wheel for who-speak-ai: filename=who_speak_ai-0.1.0-0.editable-py3-none-any.whl size=1408 sha256=a479b5ab49d61124a9b8f98227fbeb1ee72ebbd8a5691701024b0e4c1e716ae0
  Stored in directory: /tmp/pip-ephem-wheel-cache-j6keecm5/wheels/dd/37/19/34f43b94ad38dea9ffe1521f4c327b33de3a8b281f533a1285
Successfully built who-speak-ai
  Attempting uninstall: who-speak-ai
    Found existing installation: who-speak-ai 0.1.0
    Uninstalling who-speak-ai-0.1.0:
      Successfully uninstalled who-speak-ai-0.1.0


In [19]:
!python scripts/validate_vimd_protocol.py \
    --dataset-root "/kaggle/input/datasets/dullahn/vimd-dataset" \
    --output "results/data_audit/vimd_protocol_summary.json" \
    --batch-size 4096

VIMD CANONICAL PROTOCOL VALIDATED
train: 15,023 utterances, 10,291 speakers, 7,044 genuine pairs
validation: 1,898 utterances, 1,318 speakers, 879 genuine pairs
test: 2,026 utterances, 1,344 speakers, 1,046 genuine pairs
total utterances: 18,947
manifest SHA-256: ed7b764c6aaab2ba2c4ec95edadab19fd640ebca72aa06da3d36cbf93fc4747f
output: /kaggle/working/Who-Speak-AI/model/Thanh2/results/data_audit/vimd_protocol_summary.json


# 20260820

In [1]:
from pathlib import Path
import subprocess

repository = Path("/kaggle/working/Who-Speak-AI")
repository_url = "https://github.com/beaver-felix/Who-Speak-AI.git"

if repository.exists():
    subprocess.run(
        [
            "git",
            "-C",
            str(repository),
            "pull",
            "--ff-only",
            "origin",
            "thanhDT",
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "thanhDT",
            repository_url,
            str(repository),
        ],
        check=True,
    )

print("REPOSITORY READY")

Cloning into '/kaggle/working/Who-Speak-AI'...


REPOSITORY READY


In [2]:
%cd /kaggle/working/Who-Speak-AI/model/Thanh2
!python -m pip install -e ".[data]"

/kaggle/working/Who-Speak-AI/model/Thanh2
Obtaining file:///kaggle/working/Who-Speak-AI/model/Thanh2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for who-speak-ai (pyproject.toml) ... done
  Created wheel for who-speak-ai: filename=who_speak_ai-0.1.0-0.editable-py3-none-any.whl size=1408 sha256=918de6a1f253058917b23b2edf78a1987f13c9dbe0470c5fbef718b957943b62
  Stored in directory: /tmp/pip-ephem-wheel-cache-wazibl59/wheels/dd/37/19/34f43b94ad38dea9ffe1521f4c327b33de3a8b281f533a1285
Successfully built who-speak-ai


In [3]:
!python scripts/validate_vimd_protocol.py \
    --dataset-root "/kaggle/input/datasets/dullahn/vimd-dataset" \
    --output "results/data_audit/vimd_protocol_summary.json" \
    --batch-size 4096

VIMD CANONICAL PROTOCOL VALIDATED
train: 15,023 utterances, 10,291 speakers, 7,044 genuine pairs
validation: 1,898 utterances, 1,318 speakers, 879 genuine pairs
test: 2,026 utterances, 1,344 speakers, 1,046 genuine pairs
total utterances: 18,947
manifest SHA-256: ed7b764c6aaab2ba2c4ec95edadab19fd640ebca72aa06da3d36cbf93fc4747f
output: /kaggle/working/Who-Speak-AI/model/Thanh2/results/data_audit/vimd_protocol_summary.json


In [4]:
from pathlib import Path
from shutil import copy2

source = Path(
    "/kaggle/working/Who-Speak-AI/model/Thanh2/"
    "results/data_audit/vimd_protocol_summary.json"
)
destination = Path("/kaggle/working/vimd_protocol_summary.json")

copy2(source, destination)
print(destination, destination.stat().st_size)

/kaggle/working/vimd_protocol_summary.json 1175


### Selected ViMD Canonical Protocol

The canonical protocol preserves source Train and Test unchanged and excludes
two contaminated source-Validation rows belonging to `spk_73_0186` and
`spk_76_0219`.

Canonical counts:

- Train: 15,023 utterances, 10,291 speakers, 7,044 genuine pairs
- Validation: 1,898 utterances, 1,318 speakers, 879 genuine pairs
- Test: 2,026 utterances, 1,344 speakers, 1,046 genuine pairs

The exclusions remove Validation/Test speaker leakage without reducing genuine
pair capacity. The canonical manifest SHA-256 is
`ed7b764c6aaab2ba2c4ec95edadab19fd640ebca72aa06da3d36cbf93fc4747f`.